# EQM — On-the-fly end‑to‑end (MLP / CNN1D / ResNet50)

Notebook único para treinar **on-the-fly** (sem caches de imagem) em BP/CC/MF;
inclui **ProteinLoss** e **BCEWithLogitsLoss** (com `pos_weight` por IC opcional),
pré-processamentos (**equalização 1D**, **dilatação**, **enhance**) e **early stopping**.

**Fluxo**:
1. Monte o Drive e ajuste caminhos em **Config**.
2. Escolha domínio, modelo, loss e preprocess no dicionário **CFG**.
3. Rode o bloco do modelo desejado (MLP, CNN1D, ResNet50).
4. Ao final, salvamos o melhor checkpoint e avaliamos (Fmax, Fmax*, wFmax, Smin, AuPRC, IAuPRC).

> Observação: a avaliação usa **exatamente** suas funções `generate_ontology`, `propagate_preds` e `evaluate` (reimplementadas aqui com o mesmo comportamento), e adiciona uma função de _wrapper_ que retorna também um dicionário de métricas.

In [ ]:
# === Ambiente / Drive ===
import os, sys, math, json, time, random, re
import numpy as np
import pandas as pd
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)
    %pip -q install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu121  || True

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms

import cv2
from PIL import Image, ImageEnhance, ImageFilter

print('torch:', torch.__version__, '| torchvision:', torchvision.__version__)
print('CUDA disponível?', torch.cuda.is_available())

In [ ]:
# === Config ===
PROJECT_DIR = Path('/content/drive/MyDrive/ppi_function_pred_project_final_v2')  # ajuste se necessário
DATA_ROOT   = Path('/content/drive/MyDrive/dados_eqm/dados')                     # ppi.csv, go.obo, *_train.csv, *_val.csv, *_ic.csv
SAVE_DIR    = Path('/content/drive/MyDrive/eqm_runs')                            # onde salvar checkpoints/saídas
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CFG = dict(
    domain='bp',                 # 'bp' | 'cc' | 'mf'
    model='resnet',              # 'mlp' | 'cnn1d' | 'resnet'
    loss_name='protein',         # 'protein' | 'bce'
    pos_weight_mode='none',      # para BCE: 'none' | 'ic'  (usa IC como pos_weight)

    # Preprocess (aplicado ao VETOR 1D antes do pooling)
    equalize=True,               # equalização 1D do vetor de pontuações
    pooling='max',               # 'max' | 'avg'
    out_size=1024,                # 224 | 448 | 896

    # Para imagens (ResNet)
    dilate=False,                # dilatação morfológica 2D na imagem
    enhance=False,               # realce (unsharp + contraste)

    # Treino
    batch_size=32,
    lr=1e-4,
    max_epochs=200,
    patience=25,
    num_workers=0,

    # ResNet50 — controle de descongelamento
    resnet_unfreeze='all',     # 'none' (só FC) | 'last2' (layer4 + fc) | 'all'
)

# Mapeamento domínio → arquivos
DOM_INFO = {
    'bp': dict(type='biological_process',   root='GO:0008150'),
    'cc': dict(type='cellular_component',   root='GO:0005575'),
    'mf': dict(type='molecular_function',   root='GO:0003674'),
}

assert DATA_ROOT.exists(), f'DATA_ROOT não encontrado: {DATA_ROOT}'
assert PROJECT_DIR.exists(), f'PROJECT_DIR não encontrado: {PROJECT_DIR}'

In [ ]:
import multiprocessing as mp
try:
    mp.set_start_method('spawn', force=True)
except RuntimeError:
    pass  # já estava definido


#UTILS AND DATALOADERS

In [ ]:
# === Loss personalizada (ProteinLoss) ===
class ProteinLoss(nn.Module):
    """
    Implementação da loss usada em SUPERMAGOv2.
    weight_tensor deve ter mesmo comprimento do vetor-rótulo
    e estar alinhado à ordem das colunas de labels.
    """
    def __init__(self, weight_tensor, device='cuda'):
        super().__init__()
        self.weight_tensor = torch.as_tensor(weight_tensor, dtype=torch.float32, device=device)

    def forward(self, y_pred, y_true):
        sig_y_pred = torch.sigmoid(y_pred)
        ce   = self._multilabel_categorical_crossentropy(y_pred, y_true)
        go_f = self._weighted_f1_loss(sig_y_pred, y_true, centric='go')
        pr_f = self._weighted_f1_loss(sig_y_pred, y_true, centric='protein')
        return ce * go_f * pr_f

    def _multilabel_categorical_crossentropy(self, y_pred, y_true):
        y_pred = (1 - 2 * y_true) * y_pred
        y_pred_neg = y_pred - y_true * 1e16
        y_pred_pos = y_pred - (1 - y_true) * 1e16
        zeros = torch.zeros_like(y_pred[..., :1])
        y_pred_neg = torch.cat([y_pred_neg, zeros], dim=-1)
        y_pred_pos = torch.cat([y_pred_pos, zeros], dim=-1)
        neg_loss = torch.logsumexp(y_pred_neg, dim=-1)
        pos_loss = torch.logsumexp(y_pred_pos, dim=-1)
        return torch.mean(neg_loss + pos_loss)

    def _weighted_f1_loss(self, y_pred, y_true, beta=1.0, centric='protein'):
        dim = 1 if centric == 'protein' else 0
        w = self.weight_tensor
        tp = torch.sum(y_true * y_pred * w, dim=dim)
        fp = torch.sum((1 - y_true) * y_pred * w, dim=dim)
        fn = torch.sum(y_true * (1 - y_pred) * w, dim=dim)
        precision = tp / (tp + fp + 1e-16)
        recall    = tp / (tp + fn + 1e-16)
        f1 = (1 + beta**2) * precision * recall / (beta**2 * precision + recall + 1e-16)
        f1 = torch.where(torch.isnan(f1), torch.zeros_like(f1), f1)
        return 1 - torch.mean(f1)

# === Utils de equalização 1D (sem min-max global) ===
def minmax_scale(v, eps=1e-8):
    v = v.astype(np.float32)
    vmin, vmax = float(np.min(v)), float(np.max(v))
    if vmax - vmin < eps: return np.zeros_like(v, dtype=np.float32)
    return (v - vmin) / (vmax - vmin + eps)

def hist_equalize_1d(v, nbins=256):
    v = v.astype(np.float32)
    vmn, vmx = float(np.min(v)), float(np.max(v))
    if vmx > 1.0 or vmn < 0.0:
        v = minmax_scale(v)
    if v.size == 0: return v
    x    = np.clip(np.round(v * (nbins - 1)), 0, nbins - 1).astype(int)
    hist = np.bincount(x, minlength=nbins).astype(np.float32)
    cdf  = np.cumsum(hist); cdf /= cdf[-1] if cdf[-1] > 0 else 1.0
    return cdf[x].astype(np.float32)

# === Ontologia + avaliação (EXATOS ao que você enviou, com um wrapper para retornar dict) ===
def propagate_preds(predictions, ontologies_names, ontology):
    ont_n = ontologies_names.tolist()
    list_of_parents = []
    for idx_term in range(len(ont_n)):
        this_list_of_parents = []
        for parent in ontology[ont_n[idx_term]]['ancestors']:
            this_list_of_parents.append(ont_n.index(parent))
        list_of_parents.append(list(set(this_list_of_parents)))
    for idx_protein in range(len(predictions)):
        for idx_term in range(len(ont_n)):
            for idx_parent in list_of_parents[idx_term]:
                predictions[idx_protein, idx_parent] = max(predictions[idx_protein, idx_parent], predictions[idx_protein, idx_term])
    return predictions

def evaluate(preds, gt, ontologies_names, ontology, ic, root):
    import math
    preds = propagate_preds(preds, ontologies_names, ontology)
    wfmax = 0; fmax = 0; fmax_s = 0; smin = 1e100
    pr_arr, rc_arr = [], []
    for tau in np.linspace(0, 1, 101):
        wpr = wrc = num_prot_w = 0
        pr_s = rc_s = num_prot_s = 0
        pr_n = rc_n = num_prot_n = 0
        ru = mi = 0
        for i, pred in enumerate(preds):
            protein_pred = set(ontologies_names[pred >= tau].tolist())
            protein_gt   = set(ontologies_names[gt[i] == 1].tolist())

            ic_pred = sum(ic[q] for q in protein_pred)
            ic_gt   = sum(ic[q] for q in protein_gt)
            ic_intersect = sum(ic[q] for q in protein_pred.intersection(protein_gt))

            # wfmax
            if ic_pred > 0:
                num_prot_w += 1
                wpr += (ic_intersect / ic_pred)
            if ic_gt > 0:
                wrc += (ic_intersect / ic_gt)

            # fmax
            if len(protein_pred) > 0:
                num_prot_n += 1
                pr_n += len(protein_pred.intersection(protein_gt)) / len(protein_pred)
            rc_n += len(protein_pred.intersection(protein_gt)) / len(protein_gt)

            # smin
            tp = protein_pred.intersection(protein_gt)
            fp = protein_pred - tp
            fn = protein_gt - tp
            for go_id in fp:
                mi += ic[go_id]
            for go_id in fn:
                ru += ic[go_id]

            # fmax_s
            if root in protein_pred:
                protein_pred.remove(root)
            protein_gt.discard(root)
            if len(protein_pred) > 0:
                num_prot_s += 1
                pr_s += len(protein_pred.intersection(protein_gt)) / len(protein_pred)
            if len(protein_gt) > 0:
                rc_s += len(protein_pred.intersection(protein_gt)) / len(protein_gt)

        # wfmax
        tau_wpr = (wpr / num_prot_w) if num_prot_w > 0 else 0
        tau_wrc = wrc / len(preds)
        if tau_wrc + tau_wpr > 0:
            tau_wfmax = (2 * tau_wpr * tau_wrc) / (tau_wpr + tau_wrc)
            wfmax = max(wfmax, tau_wfmax)

        # fmax
        tau_pr_n = (pr_n / num_prot_n) if num_prot_n > 0 else 0
        tau_rc_n = rc_n / len(preds)
        if tau_pr_n + tau_rc_n > 0:
            tau_fmax = (2 * tau_pr_n * tau_rc_n) / (tau_pr_n + tau_rc_n)
            fmax = max(fmax, tau_fmax)

        # AuPRC baseline
        pr_arr.append(tau_pr_n)
        rc_arr.append(tau_rc_n)

        # smin
        ru = ru / len(preds); mi = mi / len(preds)
        smin = min(smin, math.sqrt((ru * ru) + (mi * mi)))

        # fmax_s
        tau_pr_s = (pr_s / num_prot_s) if num_prot_s > 0 else 0
        tau_rc_s = rc_s / len(preds)
        if tau_pr_s + tau_rc_s > 0:
            tau_fmax_s = (2 * tau_pr_s * tau_rc_s) / (tau_pr_s + tau_rc_s)
            fmax_s = max(fmax_s, tau_fmax_s)

    pr_arr = np.array(pr_arr); rc_arr = np.array(rc_arr)
    sorted_index = np.argsort(rc_arr)
    rc_arr = rc_arr[sorted_index]; pr_arr = pr_arr[sorted_index]
    auprc = np.trapz(pr_arr, rc_arr)

    ipr_arr, irc_arr = [], []
    for tau in np.linspace(0, 1, 101):
        idxs = np.where(rc_arr >= tau)[0]
        if len(idxs) != 0:
            idx = idxs[0]
            irc_arr.append(tau)
            ipr_arr.append(max(pr_arr[idx:]))
    iauprc = np.trapz(ipr_arr, irc_arr)
    print('Fmax:', fmax)
    print('Fmax*:', fmax_s)
    print('wFmax:', wfmax)
    print('Smin:', smin)
    print('AuPRC:', auprc)
    print('IAuPRC:', iauprc)

def evaluate_collect(preds, gt, ont_names, ontology, ic, root):
    # Re-executa a mesma lógica de evaluate, mas retorna dict de métricas
    preds2 = propagate_preds(preds.copy(), ont_names, ontology)
    wfmax = 0; fmax = 0; fmax_s = 0; smin = 1e100
    pr_arr, rc_arr = [], []
    for tau in np.linspace(0, 1, 101):
        wpr = wrc = num_prot_w = 0
        pr_s = rc_s = num_prot_s = 0
        pr_n = rc_n = num_prot_n = 0
        ru = mi = 0
        for i, pred in enumerate(preds2):
            protein_pred = set(ont_names[pred >= tau].tolist())
            protein_gt   = set(ont_names[gt[i] == 1].tolist())
            ic_pred = sum(ic[q] for q in protein_pred)
            ic_gt   = sum(ic[q] for q in protein_gt)
            ic_intersect = sum(ic[q] for q in protein_pred.intersection(protein_gt))
            if ic_pred > 0:
                num_prot_w += 1; wpr += (ic_intersect / ic_pred)
            if ic_gt > 0:
                wrc += (ic_intersect / ic_gt)
            if len(protein_pred) > 0:
                num_prot_n += 1; pr_n += len(protein_pred.intersection(protein_gt)) / len(protein_pred)
            rc_n += len(protein_pred.intersection(protein_gt)) / len(protein_gt)
            tp = protein_pred.intersection(protein_gt)
            fp = protein_pred - tp
            fn = protein_gt - tp
            for go_id in fp: mi += ic[go_id]
            for go_id in fn: ru += ic[go_id]
            if root in protein_pred: protein_pred.remove(root)
            protein_gt.discard(root)
            if len(protein_pred) > 0:
                num_prot_s += 1; pr_s += len(protein_pred.intersection(protein_gt)) / len(protein_pred)
            if len(protein_gt) > 0:
                rc_s += len(protein_pred.intersection(protein_gt)) / len(protein_gt)
        tau_wpr = (wpr / num_prot_w) if num_prot_w > 0 else 0
        tau_wrc = wrc / len(preds2)
        if tau_wrc + tau_wpr > 0:
            wfmax = max(wfmax, (2 * tau_wpr * tau_wrc) / (tau_wpr + tau_wrc))
        tau_pr_n = (pr_n / num_prot_n) if num_prot_n > 0 else 0
        tau_rc_n = rc_n / len(preds2)
        if tau_pr_n + tau_rc_n > 0:
            fmax = max(fmax, (2 * tau_pr_n * tau_rc_n) / (tau_pr_n + tau_rc_n))
        pr_arr.append(tau_pr_n); rc_arr.append(tau_rc_n)
        ru = ru / len(preds2); mi = mi / len(preds2)
        smin = min(smin, math.sqrt((ru * ru) + (mi * mi)))
        tau_pr_s = (pr_s / num_prot_s) if num_prot_s > 0 else 0
        tau_rc_s = rc_s / len(preds2)
        if tau_pr_s + tau_rc_s > 0:
            fmax_s = max(fmax_s, (2 * tau_pr_s * tau_rc_s) / (tau_pr_s + tau_rc_s))
    pr_arr = np.array(pr_arr); rc_arr = np.array(rc_arr)
    idx = np.argsort(rc_arr); rc_arr = rc_arr[idx]; pr_arr = pr_arr[idx]
    auprc = np.trapz(pr_arr, rc_arr)
    ipr_arr, irc_arr = [], []
    for tau in np.linspace(0, 1, 101):
        jj = np.where(rc_arr >= tau)[0]
        if len(jj) != 0:
            irc_arr.append(tau); ipr_arr.append(max(pr_arr[jj[0]:]))
    iauprc = np.trapz(ipr_arr, irc_arr)
    return dict(fmax=fmax, fmax_star=fmax_s, wfmax=wfmax, smin=smin, auprc=auprc, iauprc=iauprc)





In [ ]:
def get_ancestors(ontology, term):
    list_of_terms = []
    list_of_terms.append(term)
    data = []

    while len(list_of_terms) > 0:
        new_term = list_of_terms.pop(0)

        if new_term not in ontology:
            break
        data.append(new_term)
        for parent_term in ontology[new_term]['parents']:
            if parent_term in ontology:
                list_of_terms.append(parent_term)

    return data

def generate_ontology(file, specific_space=False, name_specific_space=''):
    ontology = {}
    gene = {}
    flag = False
    with open(file) as f:
        for line in f.readlines():
            line = line.replace('\n','')
            if line == '[Term]':
                if 'id' in gene:
                    ontology[gene['id']] = gene
                gene = {}
                gene['parents'], gene['alt_ids'] = [], []
                flag = True

            elif line == '[Typedef]':
                flag = False

            else:
                if not flag:
                    continue
                items = line.split(': ')
                if items[0] == 'id':
                    gene['id'] = items[1]
                elif items[0] == 'alt_id':
                    gene['alt_ids'].append(items[1])
                elif items[0] == 'namespace':
                    if specific_space:
                        if name_specific_space == items[1]:
                            gene['namespace'] = items[1]
                        else:
                            gene = {}
                            flag = False
                    else:
                        gene['namespace'] = items[1]
                elif items[0] == 'is_a':
                    gene['parents'].append(items[1].split(' ! ')[0])
                elif items[0] == 'name':
                    gene['name'] = items[1]
                elif items[0] == 'is_obsolete':
                    gene = {}
                    flag = False

        key_list = list(ontology.keys())
        for key in key_list:
            ontology[key]['ancestors'] = get_ancestors(ontology, key)
            for alt_ids in ontology[key]['alt_ids']:
                ontology[alt_ids] = ontology[key]

        for key, value in ontology.items():
            if 'children' not in value:
                value['children'] = []
            for p_id in value['parents']:
                if p_id in ontology:
                    if 'children' not in ontology[p_id]:
                        ontology[p_id]['children'] = []
                    ontology[p_id]['children'].append(key)

    return ontology


In [ ]:
# === Construção do dicionário PPI e Datasets ===
'''
def build_relations_dict(ppi_csv: Path, training_ids):
    df_ppi = pd.read_csv(ppi_csv)
    # Assume colunas: [protA, protB, score]
    u = df_ppi.iloc[:, 0].astype(str).values
    v = df_ppi.iloc[:, 1].astype(str).values
    s = pd.to_numeric(df_ppi.iloc[:, 2], errors='coerce').fillna(0).values.astype(np.float32)
    set_train = set(training_ids)
    id2idx    = {pid: i for i, pid in enumerate(training_ids)}
    rel = {}
    for a, b, w in zip(u, v, s):
        rel.setdefault(a, {'relation': []}); rel.setdefault(b, {'relation': []})
        if b in set_train: rel[a]['relation'].append([id2idx[b], float(w)])
        if a in set_train: rel[b]['relation'].append([id2idx[a], float(w)])
    # relação consigo mesmo
    for pid in training_ids:
        rel.setdefault(pid, {'relation': []})
        rel[pid]['relation'].append([id2idx[pid], 1.0])
    return rel, id2idx
'''

from pathlib import Path
import pandas as pd

def build_relations_dict(ppi_csv: Path, training_ids):
    df_ppi = pd.read_csv(ppi_csv)
    # Assume colunas: [protA, protB, score]
    u = df_ppi.iloc[:, 0].astype(str).values
    v = df_ppi.iloc[:, 1].astype(str).values
    s = pd.to_numeric(df_ppi.iloc[:, 2], errors='coerce').fillna(0).values.astype(float)

    set_train = set(training_ids)
    id2idx    = {pid: i for i, pid in enumerate(training_ids)}

    # --- DEDUP por par não-direcionado {A,B} com score = máx (ignora self-loops) ---
    edge_scores = {}  # chave: (min(A,B), max(A,B)) -> score_max
    for a, b, w in zip(u, v, s):
        if a == b:
            continue
        # normaliza a ordem p/ tratar como não-direcionado
        key = (a, b) if a < b else (b, a)
        w = float(w)
        prev = edge_scores.get(key)
        if (prev is None) or (w > prev):
            edge_scores[key] = w

    # --- Monta rel simétrico a partir das arestas únicas ---
    rel = {}
    def ensure(pid):
        if pid not in rel:
            rel[pid] = {'relation': []}

    for (a, b), w in edge_scores.items():
        ensure(a); ensure(b)
        if b in set_train:
            rel[a]['relation'].append([id2idx[b], w])
        if a in set_train:
            rel[b]['relation'].append([id2idx[a], w])

    # relação consigo mesmo (mesma dimensão: len(training_ids))
    for pid in training_ids:
        ensure(pid)
        rel[pid]['relation'].append([id2idx[pid], 1.0])

    return rel, id2idx



def pool_to_size(feat: np.ndarray, size=224, mode='max') -> np.ndarray:
    v = torch.tensor(feat, dtype=torch.float32)[None, None, :]
    k = max(1, feat.shape[0] // size)
    if mode == 'max': p = F.max_pool1d(v, k)
    else:             p = F.avg_pool1d(v, k)
    out = p.squeeze().detach().cpu().numpy().astype(np.float32)
    if out.shape[0] != size: out = np.resize(out, size).astype(np.float32)
    return out

def make_vector(pid: str, rel: dict, train_size: int, equalize: bool, pooling: str, out_size: int):
    # monta vetor de scores contra o conjunto de treino
    feat = np.zeros(train_size, dtype=np.float32)
    if pid in rel:
        for j, w in rel[pid]['relation']:
            if 0 <= j < train_size:
                feat[j] = max(feat[j], float(w))
    if equalize:
        feat = hist_equalize_1d(feat)
    return pool_to_size(feat, size=out_size, mode=pooling)



'''

def outer_diff_image(vec_224: np.ndarray, do_dilate=False, enhance=False, ksize=3) -> Image.Image:
    v  = vec_224.astype(np.float32)
    M  = np.abs(np.subtract.outer(v, v))
    mn, mx = float(M.min()), float(M.max())
    Mz = np.zeros_like(M, dtype=np.float32) if mx - mn < 1e-8 else (M - mn) / (mx - mn + 1e-8)
    img = (Mz * 255.0).astype(np.uint8)
    if do_dilate:
        k = np.ones((ksize, ksize), np.uint8)
        img = cv2.dilate(img, k, iterations=1)
    pil = Image.fromarray(img, mode='L')
    if enhance:
        # leve unsharp + aumento de contraste
        pil = pil.filter(ImageFilter.UnsharpMask(radius=1, percent=100, threshold=3))
        pil = ImageEnhance.Contrast(pil).enhance(1.1)
    return pil
'''
class VectorDataset(Dataset):
    """Para MLP/CNN1D: retorna vetores (B, D)"""
    def __init__(self, ids, labels, rel, train_ids, cfg):
        self.ids    = list(ids)
        self.labels = labels.astype(np.float32)
        self.rel    = rel
        self.train_size = len(train_ids)
        self.cfg    = cfg
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        pid = self.ids[idx]
        v   = make_vector(pid, self.rel, self.train_size, self.cfg['equalize'], self.cfg['pooling'], self.cfg['out_size'])
        x   = torch.tensor(v, dtype=torch.float32)
        y   = torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, y

class ImageDataset(Dataset):
    """Para ResNet: gera imagem on-the-fly e devolve tensor (3,H,W) normalizado"""
    def __init__(self, ids, labels, rel, train_ids, cfg):
        self.ids    = list(ids)
        self.labels = labels.astype(np.float32)
        self.rel    = rel
        self.train_size = len(train_ids)
        self.cfg    = cfg
        self.tfm    = transforms.Compose([
            transforms.Resize((cfg['out_size'], cfg['out_size'])),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5], std=[0.5]),
        ])
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        pid = self.ids[idx]
        v   = make_vector(pid, self.rel, self.train_size, self.cfg['equalize'], self.cfg['pooling'], self.cfg['out_size'])
        img = outer_diff_image(v, do_dilate=self.cfg['dilate'], enhance=self.cfg['enhance'])
        t   = self.tfm(img)              # [1,H,W]
        t3  = t.repeat(3,1,1)            # [3,H,W]
        y   = torch.tensor(self.labels[idx], dtype=torch.float32)
        return t3, y




# MODELS (SETUP)

In [ ]:



# === Modelos ===
class MLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, 256),       nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        return self.net(x)

class CNN1D(nn.Module):
    def __init__(self, input_dim, num_classes, output_length=256):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm1d(64)
        self.pool  = nn.MaxPool1d(2)
        self.adapt = nn.AdaptiveMaxPool1d(output_length)
        self.fc1   = nn.Linear(64 * output_length, 256)
        self.drop  = nn.Dropout(0.5)
        self.fc2   = nn.Linear(256, num_classes)
        self.relu  = nn.ReLU()
    def forward(self, x):
        x = x.unsqueeze(1)  # [B,1,L]
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.adapt(x).view(x.size(0), -1)
        x = self.drop(self.relu(self.fc1(x)))
        return self.fc2(x)

import torchvision.models as models
import torch.nn as nn
from typing import Literal

def build_resnet50(n_classes: int,
                   unfreeze: Literal['none','last2','all']='last2',
                   pretrained: bool = True) -> nn.Module:

    m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
    # substitui a FC por uma cabeça nova
    in_feats = m.fc.in_features
    m.fc = nn.Linear(in_feats, n_classes)

    # primeiro congela tudo:
    for p in m.parameters():
        p.requires_grad = False

    if unfreeze == 'none':
        # somente a FC nova treina
        for p in m.fc.parameters():
            p.requires_grad = True

    elif unfreeze == 'last2':
        # layer4 + FC treinam
        for p in m.layer4.parameters():
            p.requires_grad = True
        for p in m.fc.parameters():
            p.requires_grad = True

    elif unfreeze == 'all':
        # TODAS as camadas treinam
        for p in m.parameters():
            p.requires_grad = True

    else:
        raise ValueError(f"valor inválido para unfreeze: {unfreeze}")

    return m




# TRAIN PIPELINE (SETUP)

In [ ]:


# === Engine de treino / validação / early stopping ===
class EarlyStopper:
    def __init__(self, patience=25, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best = float('inf')
        self.count = 0
    def step(self, val):
        if val < self.best - self.min_delta:
            self.best = val; self.count = 0; return False
        self.count += 1
        return self.count > self.patience

def build_criterion(loss_name: str, ic_vector: np.ndarray, device: torch.device, pos_weight_mode='none'):
    if loss_name == 'protein':
        return ProteinLoss(weight_tensor=ic_vector, device=str(device))
    elif loss_name == 'bce':
        if pos_weight_mode == 'ic':
            pw = torch.tensor(ic_vector, dtype=torch.float32, device=device)
            return nn.BCEWithLogitsLoss(pos_weight=pw)
        else:
            return nn.BCEWithLogitsLoss()
    else:
        raise ValueError(f"loss_name desconhecido: {loss_name}")

def run_one_epoch(loader, model, criterion, optimizer, device, train=True, use_amp=True):
    model.train(mode=train)
    total = 0.0
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp and (device.type=='cuda'))
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=use_amp and (device.type=='cuda')):
                logits = model(xb); loss = criterion(logits, yb)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            with torch.no_grad():
                logits = model(xb); loss = criterion(logits, yb)
        total += float(loss.item())
    return total / max(1, len(loader))

def fit(model, train_loader, val_loader, criterion, optimizer, device, max_epochs, patience, run_tag, save_dir: Path):
    es = EarlyStopper(patience=patience)
    best_val = float('inf'); best_path = save_dir / f"{run_tag}_best.pt"
    t0 = time.time()
    for ep in range(1, max_epochs+1):
        tr = run_one_epoch(train_loader, model, criterion, optimizer, device, train=True)
        vl = run_one_epoch(val_loader,   model, criterion, optimizer, device, train=False)
        print(f"Epoch {ep:03d} | train/loss={tr:.4f} | val/loss={vl:.4f}")
        if vl < best_val:
            best_val = vl
            torch.save(model.state_dict(), best_path)
            print(f"  ✔ Saved new best: {best_path}")
        if es.step(vl):
            print(f"Early stopping (patience {patience})")
            break
    print(f"⏱️ Done in {time.time()-t0:.1f}s | Best val: {best_val:.4f}")
    return best_path

def predict_proba(loader, model, device):
    model.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs  = torch.sigmoid(logits).cpu().numpy()
            all_p.append(probs); all_y.append(yb.numpy())
    return np.vstack(all_p), np.vstack(all_y)





In [ ]:


# === Carregadores de dados (por domínio) ===
def load_domain_csvs(domain: str):
    t = (DATA_ROOT / f"{domain}_train.csv"); v = (DATA_ROOT / f"{domain}_val.csv")
    ic = (DATA_ROOT / f"{domain}_ic.csv"); go = (DATA_ROOT / 'go.obo')
    assert t.exists() and v.exists() and ic.exists() and go.exists(), 'Arquivos CSV/GO não encontrados.'
    df_tr = pd.read_csv(t); df_v = pd.read_csv(v); ic_df = pd.read_csv(ic)
    go_path = str(go)
    return df_tr, df_v, ic_df, go_path

def prepare_common(domain: str):
    df_tr, df_v, ic_df, go_path = load_domain_csvs(domain)
    terms = df_tr.columns[2:]
    ic_vec = (ic_df.set_index('terms').reindex(terms)['IC'].fillna(0).to_numpy(dtype=np.float32))
    ic_dict = ic_df.set_index('terms')['IC'].to_dict()
    ont = generate_ontology(go_path, specific_space=True, name_specific_space=DOM_INFO[domain]['type'])
    train_ids = df_tr['ID'].astype(str).values
    val_ids   = df_v['ID'].astype(str).values
    Ytr = df_tr.iloc[:, 2:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)
    Yv  = df_v.iloc[:, 2:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)
    rel, _ = build_relations_dict(DATA_ROOT/'ppi.csv', train_ids)
    return (train_ids, val_ids, Ytr, Yv, terms, ic_vec, ic_dict, ont)

def build_vector_loaders(cfg):
    domain = cfg['domain']
    train_ids, val_ids, Ytr, Yv, terms, ic_vec, ic_dict, ont = prepare_common(domain)
    ds_tr = VectorDataset(train_ids, Ytr, rel=build_relations_dict(DATA_ROOT/'ppi.csv', train_ids)[0], train_ids=train_ids, cfg=cfg)
    ds_v  = VectorDataset(val_ids,   Yv,  rel=build_relations_dict(DATA_ROOT/'ppi.csv', train_ids)[0], train_ids=train_ids, cfg=cfg)
    dl_tr = DataLoader(ds_tr, batch_size=cfg['batch_size'], shuffle=True,  num_workers=cfg['num_workers'], pin_memory=True)
    dl_v  = DataLoader(ds_v,  batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'], pin_memory=True)
    return dl_tr, dl_v, len(terms), terms, ic_vec, ic_dict, ont, train_ids, val_ids

def build_image_loaders(cfg):
    domain = cfg['domain']
    train_ids, val_ids, Ytr, Yv, terms, ic_vec, ic_dict, ont = prepare_common(domain)
    rel = build_relations_dict(DATA_ROOT/'ppi.csv', train_ids)[0]
    ds_tr = ImageDataset(train_ids, Ytr, rel=rel, train_ids=train_ids, cfg=cfg)
    ds_v  = ImageDataset(val_ids,   Yv,  rel=rel, train_ids=train_ids, cfg=cfg)
    dl_tr = DataLoader(ds_tr, batch_size=cfg['batch_size'], shuffle=True,  num_workers=cfg['num_workers'], pin_memory=True)
    dl_v  = DataLoader(ds_v,  batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'], pin_memory=True)
    return dl_tr, dl_v, len(terms), terms, ic_vec, ic_dict, ont, train_ids, val_ids


# DATA VISUALIZATION AND SANITY CHECKS

In [ ]:
# === (Opcional) Pré-visualizar algumas imagens geradas para sanity-check ===
def preview_resnet_images(cfg, n=4):
    dl_tr, dl_v, ncls, terms, ic_vec, ic_dict, ont, tr_ids, va_ids = build_image_loaders(cfg)
    xb, yb = next(iter(dl_v))
    import matplotlib.pyplot as plt
    from torchvision.utils import make_grid
    grid = make_grid(xb[:n], nrow=n, normalize=True, value_range=(-1,1))
    plt.figure(figsize=(4*n, 4))
    plt.imshow(np.transpose(grid.numpy(), (1,2,0)))
    plt.axis('off'); plt.show()

# Descomente para ver rapidamente
preview_resnet_images(CFG, n=4)

In [ ]:
# === Sanity-check: contagem de dados e shapes de batch por domínio ===
def check_domain_splits(domains=('bp','cc','mf')):
    for dom in domains:
        cfg = CFG.copy()
        cfg['domain'] = dom
        dl_tr, dl_v, ncls, terms, ic_vec, ic_dict, ont, tr_ids, va_ids = build_image_loaders(cfg)

        xb, yb = next(iter(dl_tr))
        print(f"\n[{dom.upper()}] classes={ncls} | train={len(tr_ids):,} | val={len(va_ids):,}")
        print(f"  exemplo batch train: X={tuple(xb.shape)}  y={tuple(yb.shape)}")
        xb, yb = next(iter(dl_v))
        print(f"  exemplo batch val  : X={tuple(xb.shape)}  y={tuple(yb.shape)}")

check_domain_splits(('bp','cc','mf'))


In [ ]:
# === Sanity-check: arquitetura da ResNet e camadas descongeladas ===
def resnet_sanity(model: torch.nn.Module):
    total, trainable = 0, 0
    for n, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    print(f"Parâmetros: total={total:,} | treináveis={trainable:,}")

    def flag(p): return "TRAIN" if p.requires_grad else "frozen"
    print("\nCamadas principais:")
    print(" - conv1/bn1/relu/maxpool:", flag(model.conv1.weight))
    for lname in ["layer1", "layer2", "layer3", "layer4"]:
        layer = getattr(model, lname)
        # checa o 1º bloco para exemplificar
        any_train = any(p.requires_grad for p in layer.parameters())
        print(f" - {lname}: {'TRAIN' if any_train else 'frozen'}")

    print("\nDetalhe por bloco de layer4:")
    for i, b in enumerate(model.layer4):
        any_train = any(p.requires_grad for p in b.parameters())
        print(f"   layer4[{i}]: {'TRAIN' if any_train else 'frozen'}")

# exemplo rápido:
_cfg = CFG.copy(); _cfg['model']='resnet'
dl_tr, dl_v, ncls, *_ = build_image_loaders(_cfg)
m = build_resnet50(ncls, unfreeze=_cfg['resnet_unfreeze'], pretrained=True)
resnet_sanity(m)


#RUN TRAIN AND EVALUATE

In [ ]:
# === Treinar MLP (vetor) ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = CFG.copy(); cfg['model'] = 'mlp'
dl_tr, dl_v, ncls, terms, ic_vec, ic_dict, ont, tr_ids, va_ids = build_vector_loaders(cfg)
input_dim = cfg['out_size']
model = MLP(input_dim, ncls).to(device)
criterion = build_criterion(cfg['loss_name'], ic_vec, device, pos_weight_mode=cfg['pos_weight_mode'])
optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
run_tag   = f"{cfg['domain']}_mlp_{cfg['loss_name']}_{cfg['pooling']}_{cfg['out_size']}"
best_path = fit(model, dl_tr, dl_v, criterion, optimizer, device, cfg['max_epochs'], cfg['patience'], run_tag, SAVE_DIR)

# Carrega melhor e avalia
model.load_state_dict(torch.load(best_path, map_location=device))
probs, gts = predict_proba(dl_v, model, device)
metrics = evaluate_collect(probs, gts, ont_names=terms.values, ontology=ont, ic=ic_dict, root=DOM_INFO[cfg['domain']]['root'])
print('\nMétricas (val):', metrics)
(SAVE_DIR / f"{run_tag}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', best_path)

In [ ]:
# === Treinar CNN1D (vetor) ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = CFG.copy(); cfg['model'] = 'cnn1d'
dl_tr, dl_v, ncls, terms, ic_vec, ic_dict, ont, tr_ids, va_ids = build_vector_loaders(cfg)
input_dim = cfg['out_size']
model = CNN1D(input_dim, ncls, output_length=256).to(device)
criterion = build_criterion(cfg['loss_name'], ic_vec, device, pos_weight_mode=cfg['pos_weight_mode'])
optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])
run_tag   = f"{cfg['domain']}_cnn1d_{cfg['loss_name']}_{cfg['pooling']}_{cfg['out_size']}"
best_path = fit(model, dl_tr, dl_v, criterion, optimizer, device, cfg['max_epochs'], cfg['patience'], run_tag, SAVE_DIR)

# Carrega melhor e avalia
model.load_state_dict(torch.load(best_path, map_location=device))
probs, gts = predict_proba(dl_v, model, device)
metrics = evaluate_collect(probs, gts, ont_names=terms.values, ontology=ont, ic=ic_dict, root=DOM_INFO[cfg['domain']]['root'])
print('\nMétricas (val):', metrics)
(SAVE_DIR / f"{run_tag}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', best_path)

In [ ]:



# === Treinar ResNet50 (imagens on-the-fly) ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = CFG.copy(); cfg['model'] = 'resnet'
dl_tr, dl_v, ncls, terms, ic_vec, ic_dict, ont, tr_ids, va_ids = build_image_loaders(cfg)
model = build_resnet50(ncls, unfreeze=cfg['resnet_unfreeze'], pretrained=True).to(device)
criterion = build_criterion(cfg['loss_name'], ic_vec, device, pos_weight_mode=cfg['pos_weight_mode'])
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg['lr'])
run_tag   = f"{cfg['domain']}_resnet50_{cfg['loss_name']}_{cfg['pooling']}_{cfg['out_size']}_u{cfg['resnet_unfreeze']}"
best_path = fit(model, dl_tr, dl_v, criterion, optimizer, device, cfg['max_epochs'], cfg['patience'], run_tag, SAVE_DIR)

# Carrega melhor e avalia
model.load_state_dict(torch.load(best_path, map_location=device))
probs, gts = predict_proba(dl_v, model, device)
metrics = evaluate_collect(probs, gts, ont_names=terms.values, ontology=ont, ic=ic_dict, root=DOM_INFO[cfg['domain']]['root'])
print('\nMétricas (val):', metrics)
(SAVE_DIR / f"{run_tag}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', best_path)

In [ ]:
# === (Opcional) Predição em NOVO CSV usando checkpoint salvo ===
def predict_new_csv(domain: str, model_kind: str, ckpt_path: Path, cfg_overrides: dict, csv_path: Path, out_csv: Path):
    cfg = CFG.copy(); cfg['domain'] = domain; cfg.update(cfg_overrides or {})
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    # carrega base comum
    df_tr, df_v, ic_df, go_path = load_domain_csvs(domain)
    terms = df_tr.columns[2:]
    ic_vec = (ic_df.set_index('terms').reindex(terms)['IC'].fillna(0).to_numpy(dtype=np.float32))
    # ids de treino para base do vetor
    train_ids = df_tr['ID'].astype(str).values
    rel, _    = build_relations_dict(DATA_ROOT/'ppi.csv', train_ids)
    # carrega CSV novo
    df_new = pd.read_csv(csv_path)
    ids_new = df_new['ID'].astype(str).values
    Ydummy  = df_new.iloc[:, 2:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)
    if model_kind in ('mlp','cnn1d'):
        ds = VectorDataset(ids_new, Ydummy, rel=rel, train_ids=train_ids, cfg=cfg)
    else:
        ds = ImageDataset(ids_new, Ydummy, rel=rel, train_ids=train_ids, cfg=cfg)
    dl = DataLoader(ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])
    # modelo
    ncls = len(terms)
    if model_kind == 'mlp':
        model = MLP(cfg['out_size'], ncls)
    elif model_kind == 'cnn1d':
        model = CNN1D(cfg['out_size'], ncls, output_length=256)
    else:
        model = build_resnet50(ncls, unfreeze=cfg['resnet_unfreeze'], pretrained=False)
    model.load_state_dict(torch.load(ckpt_path, map_location=device)); model.to(device)
    model.eval()
    all_probs = []
    with torch.no_grad():
        for xb, _ in dl:
            xb = xb.to(device)
            pr = torch.sigmoid(model(xb)).cpu().numpy()
            all_probs.append(pr)
    P = np.vstack(all_probs)
    out = pd.DataFrame(P, columns=terms)
    out.insert(0, 'ID', ids_new)
    out.to_csv(out_csv, index=False)
    print('Predições salvas em', out_csv)


#NOVOS EXPERIMENTOS COM IMAGENS

In [ ]:
from pathlib import Path
import gc
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

from transformers import T5EncoderModel, T5Tokenizer

# “Códigos” das últimas 3 camadas do T5 (mantendo seu padrão)
T5_L1, T5_L2, T5_L3 = '24', '23', '22'     # última, penúltima, antepenúltima
DEFAULT_MONO_LAYER = T5_L1                  # mono usa a camada '24'


In [ ]:


def sigmoid(x):
    x = 1 / (1 + np.exp(-x))
    x = np.round(x * 255)
    x = x.astype(np.uint8)  # já sai 0..255 uint8
    return x

def emb_method(matrix_embs, method):
    if method == 'mean':
        return np.mean(matrix_embs, axis=0)

def protein_embedding(X, pos, method='mean'):
    n_X = []
    last_pos = pos[0]
    cur_emb = []
    for i in range(len(X)):
        cur_pos = pos[i]
        if last_pos == cur_pos:
            cur_emb.append(X[i])
        else:
            n_X.append(emb_method(np.array(cur_emb), method))
            last_pos = cur_pos
            cur_emb = [X[i]]
    n_X.append(emb_method(np.array(cur_emb), method))
    return np.array(n_X)

def preprocess(df, subseq=1022):
    prot_list = []
    positions = []
    sequences = df.iloc[:, 1].values
    names = df.iloc[:, 0].values.tolist()
    for i in range(len(sequences)):
        len_seq = int(np.ceil(len(sequences[i]) / subseq))
        for idx in range(len_seq):
            positions.append(i)
            if idx != len_seq - 1:
                prot_list.append(sequences[i][idx * subseq : (idx + 1) * subseq])
            else:
                prot_list.append(sequences[i][idx * subseq :])
    return prot_list, positions, names
'''
def get_embeddings(seq, tokenizer, model, device):
    batch_seq = [" ".join(list(seq))]
    ids = tokenizer(batch_seq)
    input_ids = torch.tensor(ids['input_ids']).to(device)
    attention_mask = torch.tensor(ids['attention_mask']).to(device)
    with torch.no_grad():
        embedding_repr = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
    # médias por token nas 3 últimas camadas
    e1 = embedding_repr.hidden_states[-1][0].detach().cpu().numpy().mean(axis=0)
    e2 = embedding_repr.hidden_states[-2][0].detach().cpu().numpy().mean(axis=0)
    e3 = embedding_repr.hidden_states[-3][0].detach().cpu().numpy().mean(axis=0)
    return e1, e2, e3

 '''

 # === PATCH: embedding em lotes ===
import torch, numpy as np
from tqdm import tqdm

def get_embeddings_batch(seqs, tokenizer, model, device, autocast_dtype=None):
    toks = [" ".join(list(s)) for s in seqs]
    enc = tokenizer(toks, padding=True, truncation=False, return_tensors='pt')
    input_ids = enc['input_ids'].to(device, non_blocking=True)
    attention_mask = enc['attention_mask'].to(device, non_blocking=True)

    with torch.no_grad():
        if autocast_dtype is not None:
            with torch.cuda.amp.autocast(dtype=autocast_dtype):
                out = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        else:
            out = model(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)

        h1, h2, h3 = out.hidden_states[-1], out.hidden_states[-2], out.hidden_states[-3]   # [B,T,1024]
        mask  = attention_mask.unsqueeze(-1).float()
        denom = mask.sum(dim=1).clamp(min=1.0)
        e1 = (h1 * mask).sum(dim=1) / denom
        e2 = (h2 * mask).sum(dim=1) / denom
        e3 = (h3 * mask).sum(dim=1) / denom

    return (e1.detach().cpu().numpy().astype(np.float32),
            e2.detach().cpu().numpy().astype(np.float32),
            e3.detach().cpu().numpy().astype(np.float32))

def embed_and_collapse_T5_batched(seqs, pos, tokenizer, model, device, batch_size=256, autocast_dtype=None):
    n = len(seqs); D = 1024
    l1 = np.zeros((n, D), dtype=np.float32)
    l2 = np.zeros((n, D), dtype=np.float32)
    l3 = np.zeros((n, D), dtype=np.float32)

    i = 0
    pbar = tqdm(total=n, desc=f'Embed {n} seqs (adaptive)', leave=False)
    while i < n:
        bs = min(batch_size, n - i)
        try:
            e1, e2, e3 = get_embeddings_batch(seqs[i:i+bs], tokenizer, model, device, autocast_dtype)
            l1[i:i+bs], l2[i:i+bs], l3[i:i+bs] = e1, e2, e3
            i += bs
            pbar.update(bs)
            torch.cuda.empty_cache()
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs == 1: raise
            batch_size = max(1, bs // 2)  # reduz e re-tenta
            continue
    pbar.close()

    # sua função de colapso por proteína
    n1 = protein_embedding(l1, pos)
    n2 = protein_embedding(l2, pos)
    n3 = protein_embedding(l3, pos)
    return n1, n2, n3



In [ ]:


def ppi_ponderation(
    names_split,      # lista de proteínas do split (na ordem de embs_split)
    embs_split,       # np.ndarray [N_split, 1024]  (um canal: l1 OU l2 OU l3)
    rel,              # dict do build_relations_dict(...)
    train_embs,       # np.ndarray [N_train, 1024]  (pool de vizinhos = TRAIN BASE)
    alpha=0.5,
    train_name2idx=None  # dict nome->idx do TRAIN; se name ∈ dict, self é removido
):
    assert embs_split.shape[1] == 1024, "Embeddings devem ter 1024 dims"
    out = np.zeros_like(embs_split, dtype=np.float32)

    for i, name in enumerate(names_split):
        self_vec = embs_split[i]
        lst = rel.get(name, {'relation': []})['relation']
        if not lst:
            out[i] = self_vec
            continue

        idxs   = np.array([p[0] for p in lst], dtype=int)
        scores = np.array([float(p[1]) for p in lst], dtype=np.float32)

        mask = (scores > 0) & (idxs >= 0) & (idxs < train_embs.shape[0])

        # auto drop-self no TRAIN
        if train_name2idx is not None and name in train_name2idx:
            self_j = train_name2idx[name]
            mask = mask & (idxs != self_j)

        if not mask.any():
            out[i] = self_vec
            continue

        idxs   = idxs[mask]
        scores = scores[mask]
        ssum   = scores.sum()
        if ssum <= 0:
            out[i] = self_vec
            continue

        w = scores / ssum
        acc = np.zeros(1024, dtype=np.float32)
        for ww, j in zip(w, idxs):
            acc += float(ww) * train_embs[j]

        out[i] = float(alpha) * self_vec + float(1.0 - alpha) * acc

    return out


In [ ]:


def reshape_32(vec_uint8):
    """(1024,) uint8 -> (32, 32) uint8"""
    return np.asarray(vec_uint8, dtype=np.uint8).reshape(32, 32)

def to_rgb(ch1_uint8, ch2_uint8, ch3_uint8):
    """3×(1024,) uint8 -> (32,32,3) uint8"""
    return np.stack([reshape_32(ch1_uint8),
                     reshape_32(ch2_uint8),
                     reshape_32(ch3_uint8)], axis=-1)


'''
def save_images_from_embs(names, n1, n2, n3, out_mode='rgb', save_dir='../embs', save_raw_embs=True):
    out_dir = Path(save_dir); out_dir.mkdir(parents=True, exist_ok=True)
    l1_name, l2_name, l3_name = T5_L1, T5_L2, T5_L3
    for i, nm in enumerate(tqdm(names, desc='Save images/vecs')):
        if save_raw_embs:
            np.save(out_dir/f'{nm}-{l1_name}.npy', n1[i])
            np.save(out_dir/f'{nm}-{l2_name}.npy', n2[i])
            np.save(out_dir/f'{nm}-{l3_name}.npy', n3[i])
        u1 = sigmoid(n1[i]); u2 = sigmoid(n2[i]); u3 = sigmoid(n3[i])
        if out_mode == 'mono':
            np.save(out_dir/f'img-{nm}-{l1_name}.npy', reshape_32(u1))  # usa '24'
        elif out_mode == 'rgb':
            np.save(out_dir/f'img-{nm}-RGB.npy', to_rgb(u1, u2, u3))
        else:
            raise ValueError("out_mode inválido ('mono'|'rgb')")
'''

from concurrent.futures import ThreadPoolExecutor, as_completed

def _save_one(out_dir, nm, out_mode, n1i, n2i, n3i, save_raw):
    l1_name, l2_name, l3_name = T5_L1, T5_L2, T5_L3
    tasks = []
    if save_raw:
        tasks += [
            (out_dir/f'{nm}-{l1_name}.npy', n1i),
            (out_dir/f'{nm}-{l2_name}.npy', n2i),
            (out_dir/f'{nm}-{l3_name}.npy', n3i),
        ]
    u1 = sigmoid(n1i); u2 = sigmoid(n2i); u3 = sigmoid(n3i)
    if out_mode == 'mono':
        arr = reshape_32(u1)  # camada '24'
        tasks.append((out_dir/f'img-{nm}-{T5_L1}.npy', arr))
    else:
        arr = to_rgb(u1,u2,u3)
        tasks.append((out_dir/f'img-{nm}-RGB.npy', arr))

    for pth, arr in tasks:
        np.save(pth, arr)

def save_images_from_embs(names, n1, n2, n3, out_mode='rgb', save_dir=PROJECT_DIR/'embs', save_raw_embs=False, max_workers=8):
    out_dir = Path(save_dir); out_dir.mkdir(parents=True, exist_ok=True)
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = []
        for i, nm in enumerate(names):
            futures.append(ex.submit(_save_one, out_dir, nm, out_mode, n1[i], n2[i], n3[i], save_raw_embs))
        for _ in tqdm(as_completed(futures), total=len(futures), desc='Save images/vecs', leave=False):
            pass


In [ ]:


from transformers import T5EncoderModel, T5Tokenizer
import torch

def load_t5(precision: str = 'bf16', device=None):
    """
    precision: 'bf16' | 'fp16' | 'fp32' | 'int8'
    - L4: use 'bf16' (recomendado). 'fp16' também funciona.
    - 'int8' precisa bitsandbytes e device_map='auto'.
    """
    device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_name = 'Rostlab/prot_t5_xl_half_uniref50-enc'
    tok = T5Tokenizer.from_pretrained(model_name)

    if precision == 'int8':
        # pip install -q bitsandbytes accelerate  (se quiser usar)
        model = T5EncoderModel.from_pretrained(model_name, load_in_8bit=True, device_map='auto')
        autocast_dtype = None  # já está quantizado
    else:
        dtype = {'fp32': torch.float32, 'fp16': torch.float16, 'bf16': torch.bfloat16}[precision]
        model = T5EncoderModel.from_pretrained(model_name, torch_dtype=dtype).to(device)
        autocast_dtype = dtype

    model.eval()
    return tok, model, device, autocast_dtype


In [ ]:


# === BLOCO 6 (PATCH COM PATHS) ===
import shutil
import os, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.backends.cuda.matmul.allow_tf32 = True


def run_embedding_to_images_with_ppi_fast(
    ont: str,
    data_root: Path = DATA_ROOT,
    final_dir: Path = PROJECT_DIR / 'embs',
    alpha: float = 0.5,
    out_mode: str = 'rgb',
    apply_ppi_on_train: bool = True,
    batch_size: int = 64,
    save_raw_embs: bool = False
):
    tr_csv = data_root / f'{ont}_train.csv'
    va_csv = data_root / f'{ont}_val.csv'
    te_csv = data_root / f'{ont}_test.csv'
    ppi_csv = data_root / 'ppi.csv'
    for p in [tr_csv, va_csv, te_csv, ppi_csv]:
        assert p.exists(), f'Arquivo não encontrado: {p}'

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Diretório temporário local (rápido)
    tmp_dir = Path('/content/embs_tmp')
    if tmp_dir.exists(): shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    # Dados
    train_df = pd.read_csv(tr_csv)
    val_df   = pd.read_csv(va_csv)
    test_df  = pd.read_csv(te_csv)

    train, pos_tr, names_tr = preprocess(train_df)
    val,   pos_v,  names_v  = preprocess(val_df)
    test,  pos_t,  names_t  = preprocess(test_df)

    # Modelo
    model_path = 'Rostlab/prot_t5_xl_half_uniref50-enc'
    tokenizer = T5Tokenizer.from_pretrained(model_path)
    tok, model, device, autocast_dtype = load_t5(precision='fp16') # L4: bf16
    model.eval()

    # PPI
    training_ids = train_df['ID'].astype(str).values
    rel, _  = build_relations_dict(ppi_csv, training_ids)
    name2idx_tr  = {nm: i for i, nm in enumerate(names_tr)}

    # TRAIN (BASE)
    n1_tr_b, n2_tr_b, n3_tr_b = embed_and_collapse_T5_batched(
    train, pos_tr, tok, model, device,
    batch_size=256,                 # pode tentar 256–384 na L4
    autocast_dtype=autocast_dtype
)

    if apply_ppi_on_train:
        n1_tr = ppi_ponderation(names_tr, n1_tr_b, rel, n1_tr_b, alpha=alpha, train_name2idx=name2idx_tr)
        n2_tr = ppi_ponderation(names_tr, n2_tr_b, rel, n2_tr_b, alpha=alpha, train_name2idx=name2idx_tr)
        n3_tr = ppi_ponderation(names_tr, n3_tr_b, rel, n3_tr_b, alpha=alpha, train_name2idx=name2idx_tr)
    else:
        n1_tr, n2_tr, n3_tr = n1_tr_b, n2_tr_b, n3_tr_b

    save_images_from_embs(names_tr, n1_tr, n2_tr, n3_tr, out_mode=out_mode, save_dir=tmp_dir, save_raw_embs=save_raw_embs, max_workers=8)

    # VAL
    n1_v_b, n2_v_b, n3_v_b = embed_and_collapse_T5_batched(val, pos_v, tokenizer, model, device, batch_size)
    n1_v = ppi_ponderation(names_v, n1_v_b, rel, n1_tr_b, alpha=alpha, train_name2idx=name2idx_tr)
    n2_v = ppi_ponderation(names_v, n2_v_b, rel, n2_tr_b, alpha=alpha, train_name2idx=name2idx_tr)
    n3_v = ppi_ponderation(names_v, n3_v_b, rel, n3_tr_b, alpha=alpha, train_name2idx=name2idx_tr)
    save_images_from_embs(names_v, n1_v, n2_v, n3_v, out_mode=out_mode, save_dir=tmp_dir, save_raw_embs=save_raw_embs, max_workers=8)

    # TEST
    n1_t_b, n2_t_b, n3_t_b = embed_and_collapse_T5_batched(test, pos_t, tokenizer, model, device, batch_size)
    n1_t = ppi_ponderation(names_t, n1_t_b, rel, n1_tr_b, alpha=alpha, train_name2idx=name2idx_tr)
    n2_t = ppi_ponderation(names_t, n2_t_b, rel, n2_tr_b, alpha=alpha, train_name2idx=name2idx_tr)
    n3_t = ppi_ponderation(names_t, n3_t_b, rel, n3_tr_b, alpha=alpha, train_name2idx=name2idx_tr)
    save_images_from_embs(names_t, n1_t, n2_t, n3_t, out_mode=out_mode, save_dir=tmp_dir, save_raw_embs=save_raw_embs, max_workers=8)

    # move de /content para o Drive (1 operação de árvore)
    final_dir = Path(final_dir)
    final_dir.mkdir(parents=True, exist_ok=True)
    for p in tmp_dir.iterdir():
        shutil.move(str(p), str(final_dir / p.name))
    shutil.rmtree(tmp_dir)

    del model; gc.collect(); torch.cuda.empty_cache()
    return (names_tr, names_v, names_t)


In [ ]:

'''
class PrecomputedImageDataset32(Dataset):
    """
    Lê imagens .npy geradas (32x32):
      - RGB:  'img-<NAME>-RGB.npy'        shape (32,32,3) uint8
      - Mono: 'img-<NAME>-24.npy' (ou camada desejada) shape (32,32) uint8
    Retorna tensor [3,32,32] em [0,1] (ToTensor).
    """
    def __init__(self, ids, labels, image_dir, image_mode='rgb', layer_code=DEFAULT_MONO_LAYER):
        self.ids   = list(ids)
        self.y     = labels.astype(np.float32)
        self.dir   = Path(image_dir)
        self.mode  = image_mode
        self.layer = str(layer_code)

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]
        if self.mode == 'rgb':
            arr = np.load(self.dir / f'img-{name}-RGB.npy')   # (32,32,3) uint8
            pil = Image.fromarray(arr, mode='RGB')
        else:
            arr = np.load(self.dir / f'img-{name}-{self.layer}.npy')  # (32,32) uint8
            pil = Image.fromarray(arr, mode='L').convert('RGB')       # duplica p/ 3 canais
        # ToTensor sem Normalize/Resize: [0,1] float32, shape [3,32,32]
        x = torch.from_numpy(np.array(pil)).permute(2,0,1).float() / 255.0
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return x, y

'''

# PATCH 1 — substitui seu Dataset por este


'''

def build_precomputed_image_loaders32(cfg, image_dir='../embs', image_mode='rgb', layer_code=DEFAULT_MONO_LAYER):
    domain = cfg['domain']
    df_tr, df_v, ic_df, go_path = load_domain_csvs(domain)
    terms = df_tr.columns[2:]
    ic_vec = (ic_df.set_index('terms').reindex(terms)['IC'].fillna(0).to_numpy(dtype=np.float32))
    ic_dict = ic_df.set_index('terms')['IC'].to_dict()
    ont = generate_ontology(go_path, specific_space=True, name_specific_space=DOM_INFO[domain]['type'])

    train_ids = df_tr['ID'].astype(str).values
    val_ids   = df_v['ID'].astype(str).values
    Ytr = df_tr.iloc[:, 2:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)
    Yv  = df_v.iloc[:, 2:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)

    ds_tr = PrecomputedImageDataset32(train_ids, Ytr, image_dir=image_dir,
                                      image_mode=image_mode, layer_code=layer_code)
    ds_v  = PrecomputedImageDataset32(val_ids,   Yv,  image_dir=image_dir,
                                      image_mode=image_mode, layer_code=layer_code)

    dl_tr = DataLoader(ds_tr, batch_size=cfg['batch_size'], shuffle=True,
                       num_workers=cfg['num_workers'], pin_memory=True)
    dl_v  = DataLoader(ds_v,  batch_size=cfg['batch_size'], shuffle=False,
                       num_workers=cfg['num_workers'], pin_memory=True)
    return dl_tr, dl_v, len(terms), terms, ic_vec, ic_dict, ont, train_ids, val_ids
    '''

class PrecomputedImageDataset32(Dataset):
    """
    Lê imagens .npy (32x32):
      - RGB:  img-<ID>-RGB.npy  -> (32,32,3) uint8
      - Mono: img-<ID>-<layer>.npy -> (32,32) uint8
    Retorna tensor [3,32,32] em [0,1].
    """
    def __init__(self, ids, labels, image_dir, image_mode='rgb', layer_code=DEFAULT_MONO_LAYER,
                 mmap=True, retry=3, retry_delay=0.4):
        self.ids   = list(ids)
        self.y     = labels.astype(np.float32)
        self.dir   = Path(image_dir)
        self.mode  = image_mode
        self.layer = str(layer_code)
        self.mmap  = mmap
        self.retry = retry
        self.retry_delay = retry_delay

    def __len__(self):
        return len(self.ids)

    def _np_load(self, path: Path):
        last = None
        for _ in range(max(1, self.retry)):
            try:
                return np.load(path, mmap_mode='r' if self.mmap else None)
            except Exception as e:
                last = e
                time.sleep(self.retry_delay)
        raise last

    def __getitem__(self, idx):
        pid = self.ids[idx]
        if self.mode == 'rgb':
            arr = self._np_load(self.dir / f'img-{pid}-RGB.npy')      # (32,32,3) uint8
            if arr.ndim != 3:   # proteção
                arr = np.repeat(arr[...,None], 3, axis=-1)
        else:
            arr = self._np_load(self.dir / f'img-{pid}-{self.layer}.npy')  # (32,32) uint8
            if arr.ndim == 2:
                arr = np.stack([arr, arr, arr], axis=-1)              # (32,32,3)
        # [H,W,C] uint8 -> [C,H,W] float32 em [0,1]
        x = torch.from_numpy(np.ascontiguousarray(arr)).permute(2,0,1).float().div_(255.0)
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return x, y

# PATCH 2 — substitui sua função por esta (aceita dataloader_kw)

def build_precomputed_image_loaders32(
    cfg,
    image_dir='../embs',
    image_mode='rgb',
    layer_code=DEFAULT_MONO_LAYER,
    dataloader_kw: dict | None = None
):
    domain = cfg['domain']
    df_tr, df_v, ic_df, go_path = load_domain_csvs(domain)
    terms = df_tr.columns[2:]
    ic_vec = (ic_df.set_index('terms').reindex(terms)['IC'].fillna(0).to_numpy(dtype=np.float32))
    ic_dict = ic_df.set_index('terms')['IC'].to_dict()
    ont = generate_ontology(go_path, specific_space=True, name_specific_space=DOM_INFO[domain]['type'])

    train_ids = df_tr['ID'].astype(str).values
    val_ids   = df_v['ID'].astype(str).values
    Ytr = df_tr.iloc[:, 2:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)
    Yv  = df_v.iloc[:, 2:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)

    # Dataset: usa mmap e retry (bom para Drive e também ok em disco local)
    ds_tr = PrecomputedImageDataset32(train_ids, Ytr, image_dir=image_dir,
                                      image_mode=image_mode, layer_code=layer_code,
                                      mmap=True, retry=3, retry_delay=0.4)
    ds_v  = PrecomputedImageDataset32(val_ids,   Yv,  image_dir=image_dir,
                                      image_mode=image_mode, layer_code=layer_code,
                                      mmap=True, retry=3, retry_delay=0.4)

    # Defaults “seguros” para Colab/Drive (podem ser sobrescritos por dataloader_kw)
    safe_kw = dict(num_workers=0, pin_memory=False, persistent_workers=False)
    if dataloader_kw:
        safe_kw.update(dataloader_kw)

    dl_tr = DataLoader(ds_tr, batch_size=cfg['batch_size'], shuffle=True,  **safe_kw)
    dl_v  = DataLoader(ds_v,  batch_size=cfg['batch_size'], shuffle=False, **safe_kw)

    return dl_tr, dl_v, len(terms), terms, ic_vec, ic_dict, ont, train_ids, val_ids



In [ ]:


class PrecomputedImageDataset32(Dataset):
    """
    Lê imagens .npy (32x32):
      - RGB:  img-<ID>-RGB.npy  -> (32,32,3) uint8
      - Mono: img-<ID>-<layer>.npy -> (32,32) uint8
    Retorna tensor [3,32,32] em [0,1] ou normalizado (ImageNet) se normalize=True.
    """
    def __init__(self, ids, labels, image_dir, image_mode='rgb', layer_code=DEFAULT_MONO_LAYER,
                 mmap=True, retry=3, retry_delay=0.4,
                 normalize: bool = False,
                 mean=(0.485, 0.456, 0.406),
                 std=(0.229, 0.224, 0.225)):
        self.ids   = list(ids)
        self.y     = labels.astype(np.float32)
        self.dir   = Path(image_dir)
        self.mode  = image_mode
        self.layer = str(layer_code)
        self.mmap  = mmap
        self.retry = retry
        self.retry_delay = retry_delay
        self.normalize = bool(normalize)
        # guarda como tensores [3,1,1] p/ broadcast
        self.register_mean = torch.tensor(mean).view(3,1,1).float()
        self.register_std  = torch.tensor(std).view(3,1,1).float()

    def __len__(self):
        return len(self.ids)

    def _np_load(self, path: Path):
        last = None
        for _ in range(max(1, self.retry)):
            try:
                return np.load(path, mmap_mode='r' if self.mmap else None)
            except Exception as e:
                last = e
                time.sleep(self.retry_delay)
        raise last

    def __getitem__(self, idx):
        pid = self.ids[idx]
        if self.mode == 'rgb':
            arr = self._np_load(self.dir / f'img-{pid}-RGB.npy')      # (32,32,3) uint8
            if arr.ndim != 3:
                arr = np.repeat(arr[...,None], 3, axis=-1)
        else:
            arr = self._np_load(self.dir / f'img-{pid}-{self.layer}.npy')  # (32,32) uint8
            if arr.ndim == 2:
                arr = np.stack([arr, arr, arr], axis=-1)                  # (32,32,3)

        # [H,W,C] uint8 -> [C,H,W] float32 em [0,1]
        x = torch.from_numpy(np.ascontiguousarray(arr)).permute(2,0,1).float().div_(255.0)

        # normalização tipo ImageNet (só se normalize=True)
        if self.normalize:
            # evita alocação extra: (x - mean) / std
            x = (x - self.register_mean) / self.register_std

        y = torch.tensor(self.y[idx], dtype=torch.float32)
        return x, y




def build_precomputed_image_loaders32(
    cfg,
    image_dir='../embs',
    image_mode='rgb',
    layer_code=DEFAULT_MONO_LAYER,
    dataloader_kw: dict | None = None,
    normalize: bool = False,                    # <— novo
    mean=(0.485, 0.456, 0.406),                 # <— opcional
    std=(0.229, 0.224, 0.225)                   # <— opcional
):
    domain = cfg['domain']
    df_tr, df_v, ic_df, go_path = load_domain_csvs(domain)
    terms = df_tr.columns[2:]
    ic_vec = (ic_df.set_index('terms').reindex(terms)['IC'].fillna(0).to_numpy(dtype=np.float32))
    ic_dict = ic_df.set_index('terms')['IC'].to_dict()
    ont = generate_ontology(go_path, specific_space=True, name_specific_space=DOM_INFO[domain]['type'])

    train_ids = df_tr['ID'].astype(str).values
    val_ids   = df_v['ID'].astype(str).values
    Ytr = df_tr.iloc[:, 2:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)
    Yv  = df_v.iloc[:, 2:].apply(pd.to_numeric, errors='coerce').fillna(0).values.astype(np.float32)

    ds_tr = PrecomputedImageDataset32(train_ids, Ytr, image_dir=image_dir,
                                      image_mode=image_mode, layer_code=layer_code,
                                      mmap=True, retry=3, retry_delay=0.4,
                                      normalize=normalize, mean=mean, std=std)
    ds_v  = PrecomputedImageDataset32(val_ids,   Yv,  image_dir=image_dir,
                                      image_mode=image_mode, layer_code=layer_code,
                                      mmap=True, retry=3, retry_delay=0.4,
                                      normalize=normalize, mean=mean, std=std)

    safe_kw = dict(num_workers=0, pin_memory=False, persistent_workers=False)
    if dataloader_kw:
        safe_kw.update(dataloader_kw)

    dl_tr = DataLoader(ds_tr, batch_size=cfg['batch_size'], shuffle=True,  **safe_kw)
    dl_v  = DataLoader(ds_v,  batch_size=cfg['batch_size'], shuffle=False, **safe_kw)

    return dl_tr, dl_v, len(terms), terms, ic_vec, ic_dict, ont, train_ids, val_ids


In [ ]:


def show_samples(image_dir, ids, n=4, mono_layer=DEFAULT_MONO_LAYER):
    image_dir = Path(image_dir)
    print("Mostrando MONO…")
    plt.figure(figsize=(10, 3))
    for i, name in enumerate(ids[:n]):
        p = image_dir / f'img-{name}-{mono_layer}.npy'
        if not p.exists(): continue
        arr = np.load(p)  # (32,32)
        plt.subplot(1, n, i+1)
        plt.imshow(arr, cmap='gray', vmin=0, vmax=255)
        plt.title(f'{name}\nmono-{mono_layer}')
        plt.axis('off')
    plt.tight_layout(); plt.show()

    print("Mostrando RGB…")
    plt.figure(figsize=(10, 3))
    for i, name in enumerate(ids[:n]):
        p = image_dir / f'img-{name}-RGB.npy'
        if not p.exists(): continue
        arr = np.load(p)  # (32,32,3)
        plt.subplot(1, n, i+1)
        plt.imshow(arr)
        plt.title(f'{name}\nRGB')
        plt.axis('off')
    plt.tight_layout(); plt.show()


#QUICK SANITY CHECK

In [ ]:
# === PREVIEW: gera ~N imagens direto do T5 (sem PPI), salva como .npy e plota algumas ===
def generate_preview_images(
    ont: str,
    n: int = 4,
    ids: list[str] | None = None,
    out_mode: str = 'rgb',                 # 'rgb' ou 'mono'
    save_dir: Path = PROJECT_DIR / f"embs_preview_bp",
    batch_size: int = 96
):
    """
    Gera embeddings T5 -> sigmoid -> imagens 32x32 (.npy) para um subconjunto do TRAIN.
    Sem PPI (preview rápido).
    Salva em save_dir (default: PROJECT_DIR/embs_preview).
    """
    # 1) carregar TRAIN e escolher amostras
    tr_csv = DATA_ROOT / f'{ont}_train.csv'
    assert tr_csv.exists(), f'Arquivo não encontrado: {tr_csv}'
    df_tr = pd.read_csv(tr_csv)

    if ids is None:
        ids = df_tr['ID'].astype(str).head(n).tolist()
    else:
        ids = list(map(str, ids))
        # filtra o DF para conter só os IDs passados (mantém ordem dada)
    df_sample = df_tr[df_tr['ID'].astype(str).isin(ids)].copy()
    # reordena conforme ids
    df_sample['__ord__'] = pd.Categorical(df_sample['ID'].astype(str), categories=ids, ordered=True)
    df_sample = df_sample.sort_values('__ord__').drop(columns='__ord__')

    # 2) preprocess → (janelas, pos, nomes)
    seqs, pos, names = preprocess(df_sample)   # names == ids (em ordem)

    # 3) modelo T5
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_path = 'Rostlab/prot_t5_xl_half_uniref50-enc'
    tokenizer = T5Tokenizer.from_pretrained(model_path)
    model = T5EncoderModel.from_pretrained(model_path, output_hidden_states=True).to(device)
    model.eval()

    # 4) embed batched + collapse por proteína
    n1, n2, n3 = embed_and_collapse_T5_batched(seqs, pos, tokenizer, model, device, batch_size=batch_size)

    # 5) salvar imagens (sem raw embs) num diretório de preview
    save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)
    save_images_from_embs(names, n1, n2, n3, out_mode=out_mode, save_dir=save_dir,
                               save_raw_embs=False, max_workers=8)

    # 6) mostrar algumas (mono e/ou rgb)
    show_samples(save_dir, ids=names, n=min(n, len(names)), mono_layer=DEFAULT_MONO_LAYER)

    # 7) cleanup
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"Preview salvo em: {save_dir.resolve()}")
    return names, save_dir


In [ ]:


# Gera 4 imagens de preview do domínio BP, salva em PROJECT_DIR/embs_preview,
# e exibe as figuras no notebook (mono+RGB se existirem).
names_preview, preview_dir = generate_preview_images(
    ont='bp',
    n=4,
    out_mode='rgb',           # ou 'mono'
    save_dir= PROJECT_DIR / f"embs_preview_bp",
    batch_size=96
)


#EXECUTION

In [ ]:

# === GERA-IMAGENS (BP) ===
# onde salvar as imagens geradas:
EMBS_DIR = PROJECT_DIR / 'embs_bp'

names_tr, names_v, names_t = run_embedding_to_images_with_ppi_fast(
    ont='bp',
    data_root=DATA_ROOT,
    final_dir= EMBS_DIR
    alpha=0.5,
    out_mode='rgb',           # ou 'mono'
    apply_ppi_on_train=True,
    batch_size=32,            # ajuste conforme memória
    save_raw_embs=False       # economiza arquivos e I/O
)

# sanity: visualize alguns exemplos
show_samples(EMBS_DIR, ids=names_tr, n=4, mono_layer=DEFAULT_MONO_LAYER)


In [ ]:

# === GERA-IMAGENS (cc) ===
# onde salvar as imagens geradas:
EMBS_DIR = PROJECT_DIR / 'embs_cc'

names_tr, names_v, names_t = run_embedding_to_images_with_ppi_fast(
    ont='cc',
    data_root=DATA_ROOT,
    final_dir=EMBS_DIR,
    alpha=0.5,
    out_mode='rgb',           # ou 'mono'
    apply_ppi_on_train=True,
    batch_size=32,            # ajuste conforme memória
    save_raw_embs=False       # economiza arquivos e I/O
)

# sanity: visualize alguns exemplos
show_samples(EMBS_DIR, ids=names_tr, n=4, mono_layer=DEFAULT_MONO_LAYER)


In [ ]:

# === GERA-IMAGENS (MF) ===
# onde salvar as imagens geradas:
EMBS_DIR = PROJECT_DIR / 'embs_mf'

names_tr, names_v, names_t = run_embedding_to_images_with_ppi_fast(
    ont='mf',
    data_root=DATA_ROOT,
    final_dir=EMBS_DIR,
    alpha=0.5,
    out_mode='rgb',           # ou 'mono'
    apply_ppi_on_train=True,
    batch_size=32,            # ajuste conforme memória
    save_raw_embs=False       # economiza arquivos e I/O
)

# sanity: visualize alguns exemplos
show_samples(EMBS_DIR, ids=names_tr, n=4, mono_layer=DEFAULT_MONO_LAYER)


#DEBUG

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1) carrega train & ppi e monta rel/id2idx/name2idx_tr
def debug_ppi_setup(ont: str, data_root=DATA_ROOT):
    tr_csv = Path(data_root) / f'{ont}_train.csv'
    ppi_csv = Path(data_root) / 'ppi.csv'
    assert tr_csv.exists() and ppi_csv.exists(), f'Arquivos faltando: {tr_csv} / {ppi_csv}'
    df_tr = pd.read_csv(tr_csv)
    training_ids = df_tr['ID'].astype(str).values
    rel, id2idx = build_relations_dict(ppi_csv, training_ids)   # usa sua função
    name2idx_tr = {pid: i for i, pid in enumerate(training_ids)}
    pool_size   = len(training_ids)
    return df_tr, training_ids, rel, id2idx, name2idx_tr, pool_size

# 2) lista vizinhos válidos (após filtros) e top-5 pesos normalizados
def debug_neighbors(ids, rel, train_name2idx, training_pool_size, max_top=5):
    for pid in ids:
        raw = rel.get(pid, {'relation': []})['relation']
        if len(raw) == 0:
            print(f'[{pid}] sem entradas no rel (nenhuma aresta no ppi.csv).')
            continue
        idxs = np.array([j for j,_ in raw], dtype=int)
        w    = np.array([float(s) for _,s in raw], dtype=float)

        m = (w > 0) & (idxs >= 0) & (idxs < training_pool_size)
        if pid in train_name2idx:
            m &= (idxs != train_name2idx[pid])  # drop self

        if not m.any():
            print(f'[{pid}] tinha arestas, mas nenhuma válida após filtros (ou só self).')
            continue

        idxs = idxs[m]; w = w[m]
        w = w / w.sum()
        top = np.argsort(-w)[:max_top]
        print(f'[{pid}] vizinhos_válidos={len(w)} | soma_pesos_norm={w.sum():.3f}')
        for k in top:
            # tenta recuperar o nome do vizinho pelo índice do TRAIN
            inv = None
            try:
                inv = [k for k,v in train_name2idx.items() if v == idxs[k]]
                inv = inv[0] if inv else f'<idx:{idxs[k]}>'
            except:
                inv = f'<idx:{idxs[k]}>'
            print(f'   - j={idxs[k]:6d}  w={w[k]:.4f}  id_train={inv}')
        print('')

# 3) overlap entre IDs do train e do PPI (e teste com upper())
def debug_overlap(ont: str, data_root=DATA_ROOT, sample=10):
    tr_csv = Path(data_root) / f'{ont}_train.csv'
    ppi_csv = Path(data_root) / 'ppi.csv'
    df_tr = pd.read_csv(tr_csv)
    df_ppi = pd.read_csv(ppi_csv)
    train_ids = set(df_tr['ID'].astype(str).values)
    ppi_ids   = set(pd.concat([df_ppi.iloc[:,0].astype(str), df_ppi.iloc[:,1].astype(str)]).values)

    inter = train_ids & ppi_ids
    print(f'Overlap exato: {len(inter)} / {len(train_ids)} ({100*len(inter)/max(1,len(train_ids)):.1f}%)')

    # testa uppercase (útil se houver divergência de caixa)
    train_up = {x.upper() for x in train_ids}
    ppi_up   = {x.upper() for x in ppi_ids}
    inter_up = train_up & ppi_up
    if len(inter_up) > len(inter):
        print(f'Overlap após UPPER(): {len(inter_up)} / {len(train_ids)}  (sugere divergência de caixa)')

    # amostra alguns que NÃO casam
    miss = list(train_ids - ppi_ids)[:sample]
    if miss:
        print(f'Exemplos no train que não aparecem no PPI (até {sample}): {miss}')
    else:
        print('Todos os IDs do train aparecem no PPI.')

# 4) compara imagens preview vs com PPI
def compare_preview_vs_ppi(ids, preview_dir, embs_dir):
    preview_dir = Path(preview_dir); embs_dir = Path(embs_dir)
    for pid in ids:
        p_prev = preview_dir / f'img-{pid}-RGB.npy'
        p_ppi  = embs_dir    / f'img-{pid}-RGB.npy'
        if not p_prev.exists() or not p_ppi.exists():
            print(f'[{pid}] faltando arquivo: preview={p_prev.exists()} ppi={p_ppi.exists()}')
            continue
        a = np.load(p_prev).astype(np.int16)
        b = np.load(p_ppi ).astype(np.int16)
        mad = np.mean(np.abs(a - b))
        print(f'[{pid}] MAD (|preview-ppi|) = {mad:.4f}')
        fig,axs = plt.subplots(1,3,figsize=(7,3))
        axs[0].imshow(a.astype(np.uint8)); axs[0].set_title('preview'); axs[0].axis('off')
        axs[1].imshow(b.astype(np.uint8)); axs[1].set_title('com PPI'); axs[1].axis('off')
        axs[2].imshow(np.clip(np.abs(a-b),0,255).astype(np.uint8)); axs[2].set_title('|diff|'); axs[2].axis('off')
        plt.tight_layout(); plt.show()


In [ ]:
# === LISTAR IDS DO TREINO COM VIZINHOS VÁLIDOS (após filtros) ===
from pathlib import Path
import numpy as np
import pandas as pd

def list_train_ids_with_valid_neighbors(ont: str, min_deg: int = 1, top_k: int = 20):
    # 1) setup PPI + treino
    tr_csv = DATA_ROOT / f'{ont}_train.csv'
    ppi_csv = DATA_ROOT / 'ppi.csv'
    df_tr = pd.read_csv(tr_csv)
    training_ids = df_tr['ID'].astype(str).values
    rel, _ = build_relations_dict(ppi_csv, training_ids)
    name2idx_tr = {pid: i for i, pid in enumerate(training_ids)}
    pool_size = len(training_ids)

    # 2) conta vizinhos válidos por ID
    rows = []
    for pid in training_ids:
        lst = rel.get(pid, {'relation': []})['relation']
        if not lst:
            rows.append((pid, 0))
            continue
        idxs = np.array([j for j,_ in lst], dtype=int)
        w    = np.array([float(s) for _,s in lst], dtype=float)
        m = (w > 0) & (idxs >= 0) & (idxs < pool_size)
        m &= (idxs != name2idx_tr.get(pid, -1))  # drop self
        rows.append((pid, int(m.sum())))
    df = pd.DataFrame(rows, columns=['ID','deg_valid']).sort_values('deg_valid', ascending=False)
    # 3) retorna os top_k com grau >= min_deg
    df = df[df['deg_valid'] >= min_deg]
    return df.head(top_k)

# Exemplo de uso:
cand = list_train_ids_with_valid_neighbors('bp', min_deg=3, top_k=10)
print(cand)  # escolha 2-3 IDs desta lista e visualize


In [ ]:
# === PREVIEW POR ID (sem PPI) — usando SUA sigmoid ===
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

def generate_preview_for_ids(
    ont: str,
    ids: list,
    out_mode: str = 'rgb',                 # 'rgb' ou 'mono'
    save_dir: Path = PROJECT_DIR/'embs_preview_bp',
    batch_size: int = 96,
    precision: str = 'fp16',               # 'bf16' se sua GPU suportar; 'fp16' é seguro
    show: bool = True
):
    """
    Gera imagem(ens) 32x32 sem PPI só para os IDs informados.
    Salva em save_dir como 'img-<ID>-RGB.npy' (ou 'img-<ID>-24.npy' no modo mono).
    Usa a função sigmoid que você já tem no notebook.
    """
    save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)

    # 1) abrir TRAIN e filtrar IDs (mantendo ordem pedida)
    tr_csv = DATA_ROOT / f'{ont}_train.csv'
    assert tr_csv.exists(), f'Arquivo não encontrado: {tr_csv}'
    df_tr = pd.read_csv(tr_csv)
    wanted = [str(x) for x in ids]
    df_sample = df_tr[df_tr['ID'].astype(str).isin(wanted)].copy()
    if df_sample.empty:
        print("Nenhum ID encontrado no train para os IDs pedidos:", wanted);
        return []

    df_sample['__ord__'] = pd.Categorical(df_sample['ID'].astype(str), categories=wanted, ordered=True)
    df_sample = df_sample.sort_values('__ord__').drop(columns='__ord__')
    names = df_sample['ID'].astype(str).tolist()

    # 2) janelar sequências
    seqs, pos, names2 = preprocess(df_sample)
    assert names2 == names, "Ordem de nomes após preprocess não bate — verifique o preprocess."

    # 3) carregar T5 na precisão desejada
    tok, model, device, autocast_dtype = load_t5(precision=precision)

    # 4) embed batched e colapsar por proteína
    n1, n2, n3 = embed_and_collapse_T5_batched(
        seqs, pos, tok, model, device,
        batch_size=batch_size, autocast_dtype=autocast_dtype
    )

    # 5) salvar imagens (sem PPI) usando SUA sigmoid
    saved = []
    for i, nm in enumerate(names):
        if out_mode == 'mono':
            u1 = sigmoid(n1[i])                   # [1024] -> uint8
            arr = reshape_32(u1)                  # (32,32) uint8
            path = save_dir / f'img-{nm}-24.npy'
            np.save(path, arr); saved.append(str(path))
        else:
            u1 = sigmoid(n1[i]); u2 = sigmoid(n2[i]); u3 = sigmoid(n3[i])
            arr = to_rgb(u1, u2, u3)              # (32,32,3) uint8
            path = save_dir / f'img-{nm}-RGB.npy'
            np.save(path, arr); saved.append(str(path))

    # 6) opcional: mostrar
    if show:
        for p in saved:
            img = np.load(p)
            plt.figure(figsize=(2.5,2.5))
            if img.ndim == 2:
                plt.imshow(img, cmap='gray', vmin=0, vmax=255)
            else:
                plt.imshow(img.astype(np.uint8))
            plt.title(Path(p).name); plt.axis('off'); plt.show()

    # cleanup
    del model; torch.cuda.empty_cache()
    return saved


In [ ]:
# gerar só para um ID específico (sem PPI) e visualizar
_ = generate_preview_for_ids(
    ont='bp',
    ids=['Q02248'],          # coloque aqui a(s) proteína(s)
    out_mode='rgb',          # ou 'mono'
    save_dir=PROJECT_DIR/'embs_preview_bp',
    batch_size=64,
    precision='fp16',
    show=True
)



In [ ]:
# 0) monte o contexto
ont = 'bp'
df_tr, training_ids, rel, id2idx, name2idx_tr, pool_size = debug_ppi_setup(ont)

# 1) ver overlap train <-> ppi (e checar caixa)
debug_overlap(ont)

# 2) ver vizinhos efetivos (após drop-self) para alguns IDs
debug_neighbors(
    ids=['Q02248'],          # coloque aqui os IDs que você quer inspecionar
    rel=rel,
    train_name2idx=name2idx_tr,
    training_pool_size=pool_size
)

# 3) comparar preview x ppi (MAD e imagens)
compare_preview_vs_ppi(
    ids=['Q02248'],
    preview_dir=PROJECT_DIR/'embs_bp_preview',
    embs_dir=PROJECT_DIR/'embs_bp'
)


In [ ]:
# 0) monte o contexto
ont = 'bp'
df_tr, training_ids, rel, id2idx, name2idx_tr, pool_size = debug_ppi_setup(ont)

# 1) ver overlap train <-> ppi (e checar caixa)
debug_overlap(ont)

# 2) ver vizinhos efetivos (após drop-self) para alguns IDs
debug_neighbors(
    ids=['Q07623'],          # coloque aqui os IDs que você quer inspecionar
    rel=rel,
    train_name2idx=name2idx_tr,
    training_pool_size=pool_size
)

# 3) comparar preview x ppi (MAD e imagens)
compare_preview_vs_ppi(
    ids=['Q07623'],
    preview_dir=PROJECT_DIR/'embs_bp_preview',
    embs_dir=PROJECT_DIR/'embs_bp'
)


In [ ]:
# 0) monte o contexto
ont = 'mf'
df_tr, training_ids, rel, id2idx, name2idx_tr, pool_size = debug_ppi_setup(ont)

# 1) ver overlap train <-> ppi (e checar caixa)
debug_overlap(ont)

In [ ]:
# === Treino ResNet50 com imagens 32x32 pré-computadas (RGB ou MONO))(BP) ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = CFG.copy()
cfg['model'] = 'resnet'
# se quiser começar mais estável:
cfg['resnet_unfreeze'] = 'all'   # 'none' | 'last2' | 'all'
cfg['max_epochs'] = 100            # 50–100 já resolve com early stopping
# (restante do cfg fica como você já usa)

image_mode  = 'rgb'                # 'rgb' ou 'mono'
mono_layer  = DEFAULT_MONO_LAYER   # se mono, usa '24'
image_dir = PROJECT_DIR/f"embs_{cfg['domain']}" # mesma pasta usada para salvar as imagens

# loaders para imagens pré-computadas 32x32 (sem resize/normalize extra)

dl_tr, dl_v, ncls, terms, ic_vec, ic_dict, ont, tr_ids, va_ids = build_precomputed_image_loaders32(
    cfg, image_dir=image_dir, image_mode=image_mode, layer_code=mono_layer
)


model = build_resnet50(
    n_classes=ncls,
    unfreeze=cfg['resnet_unfreeze'],
    pretrained=True
).to(device)

criterion = build_criterion(cfg['loss_name'], ic_vec, device, pos_weight_mode=cfg['pos_weight_mode'])
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg['lr'])

run_tag = f"{cfg['domain']}_resnet50_precomp32_{image_mode}_{cfg['loss_name']}_u{cfg['resnet_unfreeze']}"
best_path = fit(model, dl_tr, dl_v, criterion, optimizer, device,
                cfg['max_epochs'], cfg['patience'], run_tag, SAVE_DIR)

# Avaliação
model.load_state_dict(torch.load(best_path, map_location=device))
probs, gts = predict_proba(dl_v, model, device)
metrics = evaluate_collect(
    probs, gts,
    ont_names=terms.values,
    ontology=ont,
    ic=ic_dict,
    root=DOM_INFO[cfg['domain']]['root']
)
print('\nMétricas (val):', metrics)
(SAVE_DIR / f"{run_tag}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', best_path)


In [ ]:
# === Treino ResNet50 com imagens 32x32 pré-computadas (RGB ou MONO))(CC) ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = CFG.copy()
cfg['model'] = 'resnet'
cfg['domain'] = 'cc'
# se quiser começar mais estável:
cfg['resnet_unfreeze'] = 'all'   # 'none' | 'last2' | 'all'
cfg['max_epochs'] = 100            # 50–100 já resolve com early stopping
# (restante do cfg fica como você já usa)

image_mode  = 'rgb'                # 'rgb' ou 'mono'
mono_layer  = DEFAULT_MONO_LAYER   # se mono, usa '24'
image_dir = Path('/content/embs_cc') # same folder used to save the images

# loaders para imagens pré-computadas 32x32 (sem resize/normalize extra)

dl_tr, dl_v, ncls, terms, ic_vec, ic_dict, ont, tr_ids, va_ids = build_precomputed_image_loaders32(
    cfg, image_dir=image_dir, image_mode=image_mode, layer_code=mono_layer
)


model = build_resnet50(
    n_classes=ncls,
    unfreeze=cfg['resnet_unfreeze'],
    pretrained=True
).to(device)

criterion = build_criterion(cfg['loss_name'], ic_vec, device, pos_weight_mode=cfg['pos_weight_mode'])
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg['lr'])

run_tag = f"{cfg['domain']}_resnet50_precomp32_{image_mode}_{cfg['loss_name']}_u{cfg['resnet_unfreeze']}"
best_path = fit(model, dl_tr, dl_v, criterion, optimizer, device,
                cfg['max_epochs'], cfg['patience'], run_tag, SAVE_DIR)

# Avaliação
model.load_state_dict(torch.load(best_path, map_location=device))
probs, gts = predict_proba(dl_v, model, device)
metrics = evaluate_collect(
    probs, gts,
    ont_names=terms.values,
    ontology=ont,
    ic=ic_dict,
    root=DOM_INFO[cfg['domain']]['root']
)
print('\nMétricas (val):', metrics)
(SAVE_DIR / f"{run_tag}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', best_path)

In [ ]:
from pathlib import Path
import pandas as pd

def check_split_files(image_dir: Path, ont: str, mode='rgb'):
    tr = pd.read_csv(DATA_ROOT/f'{ont}_train.csv')['ID'].astype(str).tolist()
    va = pd.read_csv(DATA_ROOT/f'{ont}_val.csv')['ID'].astype(str).tolist()
    suf = '-RGB.npy' if mode=='rgb' else '-24.npy'
    miss_tr = [pid for pid in tr if not (image_dir/f'img-{pid}{suf}').exists()]
    miss_va = [pid for pid in va if not (image_dir/f'img-{pid}{suf}').exists()]
    print(f"[{ont}] train esperados: {len(tr)}, faltando: {len(miss_tr)}")
    print(f"[{ont}]  val  esperados: {len(va)}, faltando: {len(miss_va)}")
    return miss_tr, miss_va

# exemplo:
miss_tr, miss_va = check_split_files(PROJECT_DIR/'embs_cc', ont='cc', mode='rgb')


In [ ]:
from pathlib import Path
import pandas as pd

def check_split_files(image_dir: Path, ont: str, mode='rgb'):
    tr = pd.read_csv(DATA_ROOT/f'{ont}_train.csv')['ID'].astype(str).tolist()
    va = pd.read_csv(DATA_ROOT/f'{ont}_val.csv')['ID'].astype(str).tolist()
    suf = '-RGB.npy' if mode=='rgb' else '-24.npy'
    miss_tr = [pid for pid in tr if not (image_dir/f'img-{pid}{suf}').exists()]
    miss_va = [pid for pid in va if not (image_dir/f'img-{pid}{suf}').exists()]
    print(f"[{ont}] train esperados: {len(tr)}, faltando: {len(miss_tr)}")
    print(f"[{ont}]  val  esperados: {len(va)}, faltando: {len(miss_va)}")
    return miss_tr, miss_va

# exemplo:
miss_tr, miss_va = check_split_files(PROJECT_DIR/'embs_bp', ont='bp', mode='rgb')


In [ ]:
from pathlib import Path
import pandas as pd

def check_split_files(image_dir: Path, ont: str, mode='rgb'):
    tr = pd.read_csv(DATA_ROOT/f'{ont}_train.csv')['ID'].astype(str).tolist()
    va = pd.read_csv(DATA_ROOT/f'{ont}_val.csv')['ID'].astype(str).tolist()
    suf = '-RGB.npy' if mode=='rgb' else '-24.npy'
    miss_tr = [pid for pid in tr if not (image_dir/f'img-{pid}{suf}').exists()]
    miss_va = [pid for pid in va if not (image_dir/f'img-{pid}{suf}').exists()]
    print(f"[{ont}] train esperados: {len(tr)}, faltando: {len(miss_tr)}")
    print(f"[{ont}]  val  esperados: {len(va)}, faltando: {len(miss_va)}")
    return miss_tr, miss_va

# exemplo:
miss_tr, miss_va = check_split_files(PROJECT_DIR/'embs_mf', ont='mf', mode='rgb')


In [ ]:
# === Treino ResNet50 com imagens 32x32 pré-computadas (RGB ou MONO))(MF) ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = CFG.copy()
cfg['model'] = 'resnet'
cfg['domain'] = 'mf'
# se quiser começar mais estável:
cfg['resnet_unfreeze'] = 'all'   # 'none' | 'last2' | 'all'
cfg['max_epochs'] = 100            # 50–100 já resolve com early stopping
# (restante do cfg fica como você já usa)
cfg['loss_name'] = 'protein'

image_mode  = 'rgb'                # 'rgb' ou 'mono'
mono_layer  = DEFAULT_MONO_LAYER   # se mono, usa '24'
image_dir = PROJECT_DIR/f"embs_mf" # mesma pasta usada para salvar as imagens

# loaders para imagens pré-computadas 32x32 (sem resize/normalize extra)

dl_tr, dl_v, ncls, terms, ic_vec, ic_dict, ont, tr_ids, va_ids = build_precomputed_image_loaders32(
    cfg, image_dir=image_dir, image_mode=image_mode, layer_code=mono_layer, normalize=False
)


model = build_resnet50(
    n_classes=ncls,
    unfreeze=cfg['resnet_unfreeze'],
    pretrained=True
).to(device)

criterion = build_criterion(cfg['loss_name'], ic_vec, device, pos_weight_mode=cfg['pos_weight_mode'])
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=cfg['lr'])

run_tag = f"{cfg['domain']}_resnet50_precomp32_{image_mode}_{cfg['loss_name']}_u{cfg['resnet_unfreeze']}"
best_path = fit(model, dl_tr, dl_v, criterion, optimizer, device,
                cfg['max_epochs'], cfg['patience'], run_tag, SAVE_DIR)

# Avaliação
model.load_state_dict(torch.load(best_path, map_location=device))
probs, gts = predict_proba(dl_v, model, device)
metrics = evaluate_collect(
    probs, gts,
    ont_names=terms.values,
    ontology=ont,
    ic=ic_dict,
    root=DOM_INFO[cfg['domain']]['root']
)
print('\nMétricas (val):', metrics)
(SAVE_DIR / f"{run_tag}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', best_path)


#RECUPERANDO PROTEINAS DE CC FALTANTES

In [ ]:
from pathlib import Path
import pandas as pd

def list_missing_ids(ont: str, image_dir: Path, mode='rgb'):
    suf = '-RGB.npy' if mode=='rgb' else '-24.npy'
    tr = pd.read_csv(DATA_ROOT/f'{ont}_train.csv')['ID'].astype(str).tolist()
    va = pd.read_csv(DATA_ROOT/f'{ont}_val.csv')['ID'].astype(str).tolist()

    miss_tr = [pid for pid in tr if not (image_dir/f'img-{pid}{suf}').exists()]
    miss_va = [pid for pid in va if not (image_dir/f'img-{pid}{suf}').exists()]
    print(f"[{ont}] train esperados: {len(tr)}, faltando: {len(miss_tr)}")
    print(f"[{ont}]   val esperados: {len(va)}, faltando: {len(miss_va)}")
    return miss_tr, miss_va

# exemplo:
EMBS_CC = PROJECT_DIR / 'embs_cc'
missing_tr, missing_va = list_missing_ids('cc', EMBS_CC, mode='rgb')


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, torch, gc

def build_train_cache(ont: str, cache_path: Path, batch_embed=64, precision='fp16'):
    df_tr = pd.read_csv(DATA_ROOT/f'{ont}_train.csv')
    seqs_tr, pos_tr, names_tr_all = preprocess(df_tr)

    tok, model, device, autocast_dtype = load_t5(precision=precision)
    n1_tr, n2_tr, n3_tr = embed_and_collapse_T5_batched(
        seqs_tr, pos_tr, tok, model, device,
        batch_size=batch_embed, autocast_dtype=autocast_dtype
    )
    del model; gc.collect(); torch.cuda.empty_cache()

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(cache_path,
                        n1=n1_tr.astype(np.float32),
                        n2=n2_tr.astype(np.float32),
                        n3=n3_tr.astype(np.float32),
                        names=np.array(names_tr_all, dtype=object))
    print(f"✔ Cache salvo: {cache_path} | shapes: {n1_tr.shape}, {n2_tr.shape}, {n3_tr.shape}")
    return cache_path

# exemplo:
# cache_file = PROJECT_DIR / 'cache' / 'cc_train_base.npz'
# build_train_cache('cc', cache_file, batch_embed=64, precision='fp16')


In [ ]:
import os, time, gc, shutil
import numpy as np, pandas as pd, torch
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

def safe_save_npy(dst_path: Path, arr: np.ndarray, tries=3, delay=0.5):
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_local = Path('/content/tmp_save') / dst_path.name
    tmp_local.parent.mkdir(parents=True, exist_ok=True)
    last = None
    for _ in range(tries):
        try:
            np.save(tmp_local, arr)
            shutil.move(str(tmp_local), str(dst_path))
            return
        except Exception as e:
            last = e; time.sleep(delay)
    raise last

def regenerate_with_ppi_for_ids_cached(
    ont: str,
    ids: list[str],
    final_dir: Path,
    cache_npz: Path,
    alpha: float = 0.5,
    out_mode: str = 'rgb',
    batch_embed: int = 64,
    precision: str = 'fp16',
    max_workers_save: int = 8
):
    final_dir.mkdir(parents=True, exist_ok=True)

    # 1) filtrar CSVs para os IDs pedidos (train+val)
    df_tr = pd.read_csv(DATA_ROOT/f'{ont}_train.csv')
    df_va = pd.read_csv(DATA_ROOT/f'{ont}_val.csv')
    df_all = pd.concat([df_tr, df_va], ignore_index=True)
    df_sel = df_all[df_all['ID'].astype(str).isin(ids)].copy()
    if df_sel.empty:
        print("Nenhum ID solicitado está nos CSVs."); return []

    # 2) preprocess (janelas, pos, nomes compactados)
    seqs, pos, names = preprocess(df_sel)
    names = list(map(str, names))

    # 3) carregar T5 e embutir apenas os IDs alvo
    tok, model, device, autocast_dtype = load_t5(precision=precision)
    n1_b, n2_b, n3_b = embed_and_collapse_T5_batched(
        seqs, pos, tok, model, device,
        batch_size=batch_embed, autocast_dtype=autocast_dtype
    )

    # 4) carregar cache do TRAIN (memmap)
    z = np.load(cache_npz, allow_pickle=True, mmap_mode='r')
    n1_tr_base, n2_tr_base, n3_tr_base = z['n1'], z['n2'], z['n3']
    names_tr_all = z['names'].tolist()
    base_index = {pid: i for i, pid in enumerate(names_tr_all)}

    # 5) PPI relations (contra TRAIN)
    training_ids = df_tr['ID'].astype(str).values
    rel, _ = build_relations_dict(DATA_ROOT/'ppi.csv', training_ids)
    name2idx_tr = {pid: i for i, pid in enumerate(training_ids)}
    pool_size   = len(training_ids)

    # 6) mix
    def mix_one(idx, pid):
        lst = rel.get(pid, {'relation': []})['relation']
        if not lst: return n1_b[idx], n2_b[idx], n3_b[idx]
        j = np.array([t[0] for t in lst], dtype=int)
        w = np.array([t[1] for t in lst], dtype=float)
        m = (w > 0) & (j >= 0) & (j < pool_size)
        if pid in name2idx_tr: m &= (j != name2idx_tr[pid])  # drop self
        if not m.any(): return n1_b[idx], n2_b[idx], n3_b[idx]
        j = j[m]; w = w[m]; w = w / w.sum()

        # pega por índice no TRAIN (cache)
        e1 = (n1_tr_base[j] * w[:,None]).sum(axis=0)
        e2 = (n2_tr_base[j] * w[:,None]).sum(axis=0)
        e3 = (n3_tr_base[j] * w[:,None]).sum(axis=0)

        m1 = (1.0 - alpha) * n1_b[idx] + alpha * e1
        m2 = (1.0 - alpha) * n2_b[idx] + alpha * e2
        m3 = (1.0 - alpha) * n3_b[idx] + alpha * e3
        return m1, m2, m3

    n1_m = np.empty_like(n1_b); n2_m = np.empty_like(n2_b); n3_m = np.empty_like(n3_b)
    for i, pid in enumerate(names):
        n1_m[i], n2_m[i], n3_m[i] = mix_one(i, pid)

    # 7) salvar
    suf = '-RGB.npy' if out_mode=='rgb' else '-24.npy'
    def _save_one(i, pid):
        if out_mode == 'mono':
            arr = reshape_32(sigmoid(n1_m[i]))
        else:
            arr = to_rgb(sigmoid(n1_m[i]), sigmoid(n2_m[i]), sigmoid(n3_m[i]))
        safe_save_npy(final_dir/f'img-{pid}{suf}', arr)

    with ThreadPoolExecutor(max_workers=max_workers_save) as ex:
        list(as_completed([ex.submit(_save_one, i, pid) for i, pid in enumerate(names)]))

    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"✔ Regeneradas {len(names)} imagens em {final_dir}")
    return names


In [ ]:
# 1) construir cache uma vez
cache_cc = PROJECT_DIR/'cache'/'cc_train_base.npz'
build_train_cache('cc', cache_cc, batch_embed=64, precision='fp16')

# 2) listar faltantes
missing_tr, missing_va = list_missing_ids('cc', PROJECT_DIR/'embs_cc', mode='rgb')

# 3) regenerar em LOTES, agora sem re-embedar TRAIN
lote = 2000
for start in range(0, len(missing_tr), lote):
    ids = missing_tr[start:start+lote]
    regenerate_with_ppi_for_ids_cached(
        ont='cc', ids=ids, final_dir=PROJECT_DIR/'embs_cc',
        cache_npz=cache_cc, alpha=0.5, out_mode='rgb',
        batch_embed=64, precision='fp16', max_workers_save=8
    )
for start in range(0, len(missing_va), lote):
    ids = missing_va[start:start+lote]
    regenerate_with_ppi_for_ids_cached(
        ont='cc', ids=ids, final_dir=PROJECT_DIR/'embs_cc',
        cache_npz=cache_cc, alpha=0.5, out_mode='rgb',
        batch_embed=64, precision='fp16', max_workers_save=8
    )


In [ ]:
from pathlib import Path
import pandas as pd

ONT = 'mf'
MODE = 'rgb'  # ou 'mono'
SUF  = '-RGB.npy' if MODE=='rgb' else '-24.npy'

drive_dir = (PROJECT_DIR / f'embs_{ONT}')  # ex.: .../embs_cc
local_dir = Path('/content') / f'embs_{ONT}_local'  # cache local

df_tr = pd.read_csv(DATA_ROOT/f'{ONT}_train.csv')
df_va = pd.read_csv(DATA_ROOT/f'{ONT}_val.csv')
ids_tr = df_tr['ID'].astype(str).tolist()
ids_va = df_va['ID'].astype(str).tolist()
len(ids_tr), len(ids_va)


In [ ]:
import shutil, time, os
from concurrent.futures import ThreadPoolExecutor, as_completed

def _copy_one(src: Path, dst: Path, tries=3, delay=0.4):
    dst.parent.mkdir(parents=True, exist_ok=True)
    last = None
    for _ in range(tries):
        try:
            if not dst.exists():
                shutil.copyfile(src, dst)
            return True
        except Exception as e:
            last = e; time.sleep(delay)
    return False  # falhou

def stage_split_to_local(ids, drive_dir: Path, local_dir: Path, suf: str, workers=32, show_every=2000):
    drive_dir = Path(drive_dir); local_dir = Path(local_dir)
    todo = []
    for pid in ids:
        src = drive_dir / f'img-{pid}{suf}'
        dst = local_dir / f'img-{pid}{suf}'
        if not dst.exists():
            todo.append((src, dst))
    ok = skip = miss = 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = []
        for src, dst in todo:
            if not src.exists():
                miss += 1
                continue
            futs.append(ex.submit(_copy_one, src, dst))
        for i, fut in enumerate(as_completed(futs), 1):
            ok += int(fut.result())
            if i % show_every == 0:
                print(f'copiados {i}/{len(futs)}...')
    print(f'copiados={ok}, faltavam_no_drive={miss}, já_existiam={len(ids)-ok-miss}')
    return local_dir

# copie train e val para local
local_tr = stage_split_to_local(ids_tr, drive_dir, local_dir, SUF, workers=32)
local_va = stage_split_to_local(ids_va, drive_dir, local_dir, SUF, workers=32)
print('cache local pronto em:', local_dir)


In [ ]:
from pathlib import Path
import numpy as np

image_dir = Path("/content/embs_cc_local")

# 1) Conta quantos arquivos .npy existem
all_files = list(image_dir.glob("*.npy"))
print(f"Total de arquivos encontrados em {image_dir}: {len(all_files)}")

# 2) Mostra alguns exemplos de nomes de arquivos
print("Alguns exemplos de arquivos:", [f.name for f in all_files[:5]])

# 3) Carrega 1 arquivo de treino e imprime shape/dtype/range
sample_file = all_files[0]
arr = np.load(sample_file)
print(f"Exemplo: {sample_file.name} | shape={arr.shape}, dtype={arr.dtype}, min={arr.min()}, max={arr.max()}")

# 4) Verifica correspondência de IDs com CSV
df_tr, df_v, *_ = load_domain_csvs("cc")   # sua função já pronta
train_ids = set(df_tr["ID"].astype(str))
val_ids   = set(df_v["ID"].astype(str))

found_ids = {f.name.split("-")[1] for f in all_files}

missing_train = train_ids - found_ids
missing_val   = val_ids - found_ids

print(f"Proteínas de treino faltantes: {len(missing_train)}")
print(f"Proteínas de validação faltantes: {len(missing_val)}")


In [ ]:
from pathlib import Path

# === Treino ResNet50 com imagens 32x32 pré-computadas (RGB ou MONO) (mf) ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = CFG.copy()
cfg['model'] = 'resnet'
cfg['domain'] = 'mf'
cfg['resnet_unfreeze'] = 'all'   # 'none' | 'last2' | 'all'
cfg['max_epochs'] = 100
cfg['loss_name'] = 'protein'

image_mode  = 'rgb'                # 'rgb' ou 'mono'
mono_layer  = DEFAULT_MONO_LAYER   # se mono, usa '24'

# cache local com os .npy de train/val
image_dir = Path('/content/embs_mf_local') #(opcional)
#image_dir = PROJECT_DIR/f"embs_{cfg['domain']}"

# CAPTURE TUDO que o loader retorna
dl_tr, dl_v, ncls, terms, ic_vec, ic_dict, ont, tr_ids, va_ids = build_precomputed_image_loaders32(
    cfg,
    image_dir=image_dir,
    image_mode=image_mode,
    layer_code=mono_layer,
    normalize=False  # ative p/ ResNet pré-treinada no ImageNet
)

# constrói o modelo COM o ncls correto (de bp)
model = build_resnet50(
    n_classes=ncls,
    unfreeze=cfg['resnet_unfreeze'],
    pretrained=True
).to(device)

criterion = build_criterion(cfg['loss_name'], ic_vec, device, pos_weight_mode=cfg['pos_weight_mode'])
optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=cfg['lr'])

run_tag = f"{cfg['domain']}_resnet50_precomp32_{image_mode}_{cfg['loss_name']}_u{cfg['resnet_unfreeze']}"
best_path = fit(model, dl_tr, dl_v, criterion, optimizer, device,
                cfg['max_epochs'], cfg['patience'], run_tag, SAVE_DIR)

# Avaliação
model.load_state_dict(torch.load(best_path, map_location=device))
probs, gts = predict_proba(dl_v, model, device)
metrics = evaluate_collect(
    probs, gts,
    ont_names=terms.values, ontology=ont, ic=ic_dict,
    root=DOM_INFO[cfg['domain']]['root']
)
print('\nMétricas (val):', metrics)
(SAVE_DIR / f"{run_tag}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', best_path)


In [ ]:
import numpy as np

def count_null_ppi_vectors(domain: str,
                           threshold: float = 0.0,
                           ppi_min_score: float = 0.0,
                           drop_self: bool = True):
    """
    Conta quantos vetores PPI (contra o TRAIN) são nulos (todos elementos <= threshold)
    para TREINO e VALIDAÇÃO, antes e depois da equalização.

    - drop_self: remove a autocorrelação (índice do próprio ID de treino).
    - ppi_min_score: ignora interações com score <= ppi_min_score.
    - threshold: usado APENAS para decidir se o vetor é 'nulo' (tipicamente 0.0).

    OBS: hist_equalize_1d mapeia vetor constante-zero para CONSTANTE ~1.0,
         então 'depois da equalização' quase sempre retornará 0 nulos se threshold=0.0.
    """
    df_tr, df_v, _, _ = load_domain_csvs(domain)
    train_ids = df_tr['ID'].astype(str).values
    val_ids   = df_v['ID'].astype(str).values
    rel, id2idx = build_relations_dict(DATA_ROOT/'ppi.csv', train_ids)
    train_size = len(train_ids)

    def ppi_vec(pid: str) -> np.ndarray:
        v = np.zeros(train_size, dtype=np.float32)
        if pid in rel:
            self_j = id2idx.get(pid, None) if drop_self else None
            for j, w in rel[pid]['relation']:
                # drop self
                if (self_j is not None) and (j == self_j):
                    continue
                # aplica limiar de ativação no score
                if (w > ppi_min_score) and (0 <= j < train_size):
                    # agrega por máximo (como no seu make_vector)
                    if w > v[j]:
                        v[j] = float(w)
        return v

    # --- Antes da equalização
    tr_null_before = sum(1 for pid in train_ids if np.all(ppi_vec(pid) <= threshold))
    va_null_before = sum(1 for pid in val_ids   if np.all(ppi_vec(pid) <= threshold))

    # --- Depois da equalização
    tr_null_after = 0
    va_null_after = 0
    for pid in train_ids:
        v = hist_equalize_1d(ppi_vec(pid))
        if np.all(v <= threshold):
            tr_null_after += 1
    for pid in val_ids:
        v = hist_equalize_1d(ppi_vec(pid))
        if np.all(v <= threshold):
            va_null_after += 1

    res = {
        'train_size': int(len(train_ids)),
        'val_size':   int(len(val_ids)),
        'vector_length': int(train_size),
        'threshold': float(threshold),
        'ppi_min_score': float(ppi_min_score),
        'drop_self': bool(drop_self),
        'train_null_before': int(tr_null_before),
        'val_null_before':   int(va_null_before),
        'train_null_after_equalize': int(tr_null_after),
        'val_null_after_equalize':   int(va_null_after),
    }

    print(
        f"[{domain.upper()}] null vectors (<= {threshold}) "
        f"| drop_self={drop_self}, ppi_min_score>{ppi_min_score} "
        f"=> TRAIN before={tr_null_before}, after_eq={tr_null_after} "
        f"| VAL before={va_null_before}, after_eq={va_null_after} "
        f"(vector_len={train_size})"
    )
    return res



In [ ]:
# zeros exatos, sem filtrar score
count_null_ppi_vectors('bp', threshold=0.0, ppi_min_score=0.0, drop_self=True)

# só conta interações com score > 0.05 (mais restrito — costumo ver nulos aqui)
count_null_ppi_vectors('bp', threshold=0.0, ppi_min_score=0.05, drop_self=True)



In [ ]:
# zeros exatos, sem filtrar score
count_null_ppi_vectors('cc', threshold=0.0, ppi_min_score=0.0, drop_self=True)


In [ ]:
# zeros exatos, sem filtrar score
count_null_ppi_vectors('mf', threshold=0.0, ppi_min_score=0.0, drop_self=True)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

def _ppi_degree_strength_for_ids(ids, rel, id2idx, threshold=0.0, exclude_self=True):
    deg = np.zeros(len(ids), dtype=np.int32)
    strength = np.zeros(len(ids), dtype=np.float32)
    wmax = np.zeros(len(ids), dtype=np.float32)
    neff = np.zeros(len(ids), dtype=np.float32)  # 1/sum(w^2) após normalização local

    for i, pid in enumerate(ids):
        lst = rel.get(pid, {'relation': []})['relation']
        if not lst:
            continue

        # filtra e opcionalmente remove self-loop (só existe para treino)
        cur = []
        for j, w in lst:
            if exclude_self and (pid in id2idx) and (j == id2idx[pid]):
                continue
            if w > threshold:
                cur.append(float(w))

        if not cur:
            continue

        w = np.array(cur, dtype=np.float32)
        deg[i] = w.size
        strength[i] = float(w.sum())
        wmax[i] = float(w.max())

        # normaliza e calcula N_eff = 1 / sum(w_norm^2)
        wn = w / (w.sum() + 1e-12)
        neff[i] = float(1.0 / np.sum(wn * wn))

    return deg, strength, wmax, neff

def _summ(name, arr):
    q = np.nanquantile(arr.astype(np.float64), [0.0,0.05,0.25,0.5,0.75,0.95,1.0])
    return (f"{name}: "
            f"min={q[0]:.3g} | p5={q[1]:.3g} | p25={q[2]:.3g} | "
            f"p50={q[3]:.3g} | p75={q[4]:.3g} | p95={q[5]:.3g} | max={q[6]:.3g}")

def ppi_distribution_report(domain: str, threshold: float = 0.0, bins: int = 50):
    # 1) CSVs e relações contra o TRAIN
    df_tr, df_v, _, _ = load_domain_csvs(domain)
    train_ids = df_tr['ID'].astype(str).values
    val_ids   = df_v['ID'].astype(str).values
    rel, id2idx = build_relations_dict(DATA_ROOT/'ppi.csv', train_ids)

    # 2) Degree/strength/etc (exclui self no treino)
    deg_tr, str_tr, wmax_tr, neff_tr = _ppi_degree_strength_for_ids(train_ids, rel, id2idx, threshold, exclude_self=True)
    deg_va, str_va, wmax_va, neff_va = _ppi_degree_strength_for_ids(val_ids,   rel, id2idx, threshold, exclude_self=True)

    # 3) Resumos rápidos
    print(f"\n=== {domain.upper()} @ threshold={threshold} ===")
    print(f"Train proteins: {len(train_ids)} | Val proteins: {len(val_ids)}")
    print(f"Zero-degree count  — train: {(deg_tr==0).sum()} | val: {(deg_va==0).sum()}")
    print(_summ("degree(train)", deg_tr)); print(_summ("degree(val)", deg_va))
    print(_summ("strength(train)", str_tr)); print(_summ("strength(val)", str_va))
    print(_summ("max_w(train)", wmax_tr)); print(_summ("max_w(val)", wmax_va))
    print(_summ("N_eff(train)", neff_tr)); print(_summ("N_eff(val)", neff_va))

    # 4) Histogramas (grau)
    plt.figure(figsize=(7,4.2))
    m = max(deg_tr.max(), deg_va.max())
    bins_deg = np.arange(0, m+1)
    plt.hist(deg_tr, bins=bins_deg, alpha=0.6, label='train')
    plt.hist(deg_va, bins=bins_deg, alpha=0.6, label='val')
    plt.xlabel(f"degree (> {threshold})"); plt.ylabel("proteins"); plt.title(f"{domain.upper()} — degree dist")
    plt.legend(); plt.tight_layout(); plt.show()

    # 5) Cobertura vs threshold
    thr_grid = np.linspace(0.0, max(0.5, float(max(wmax_tr.max(), wmax_va.max()))), 40)
    cov_tr, cov_va = [], []
    for thr in thr_grid:
        dtr, *_ = _ppi_degree_strength_for_ids(train_ids, rel, id2idx, threshold=thr, exclude_self=True)
        dva,  *_ = _ppi_degree_strength_for_ids(val_ids,   rel, id2idx, threshold=thr, exclude_self=True)
        cov_tr.append( (dtr > 0).mean() )
        cov_va.append( (dva > 0).mean() )

    plt.figure(figsize=(7,4.2))
    plt.plot(thr_grid, cov_tr, label='train')
    plt.plot(thr_grid, cov_va, label='val')
    plt.xlabel("threshold (edge weight)"); plt.ylabel("coverage (fraction with degree>0)")
    plt.title(f"{domain.upper()} — coverage vs threshold")
    plt.ylim(0,1); plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()


    return {
        "train": {"ids": train_ids, "degree": deg_tr, "strength": str_tr, "wmax": wmax_tr, "neff": neff_tr},
        "val":   {"ids": val_ids,   "degree": deg_va, "strength": str_va, "wmax": wmax_va, "neff": neff_va},
    }

# Exemplo:
stats_bp = ppi_distribution_report('bp', threshold=0.0)
stats_cc = ppi_distribution_report('cc', threshold=0.0)
stats_mf = ppi_distribution_report('mf', threshold=0.0)